# Operating thresholds for the released full-res model (`fr_neg`) (Colab, T4 or CPU)

The released model ([`sighmon/methane-plume-se2-fullres`](https://huggingface.co/sighmon/methane-plume-se2-fullres)) is documented at one
pixel threshold, 0.5. At 0.5 it flags about a third of real plume-free 2024 scenes (round 2: 35.6 % of the 8,740 `real2024_strict`
pairs). This notebook picks **operating thresholds on 2021 val only** and reports what they do on the 2024 test split. Nothing is retrained.

**Two thresholds, one network output.** The contract is unchanged (`reflectance` 1×8×200×200 → `plume_logits` 1×1×200×200).
```
p  = sigmoid(plume_logits);  pp = avg_pool2d(p, 2)      # 20 m grid
scene_score = max(pp)                                    # stage 1: plume present?  scene_score >= T_scene
mask = (pp >= t_px) if scene_score >= T_scene else empty # stage 2: where?
```
* `T_scene`: the threshold giving 10 / 5 / 1 % false alarms on 2,000 **2021 val** plume-free pairs (same pairs as round 2).
* `t_px`: the pixel threshold with the best pixel IoU on the 193 **2021 val** plume chips (20 m grid).

Why two: the scene score is the *maximum* of ~10,000 pooled pixels, so false-alarm-calibrated values sit near 0.99. Used as a pixel
threshold, 0.9966 shrank every mask to its few most confident pixels (round 2: unseen IoU 0.03). Gating by scene and masking at a
normal pixel threshold keeps the mask quality and cuts the false alarms.

**Reported on test (2024), never used to choose anything:** scene detect and hit rates on all / seen / unseen-site plumes, the real
false-alarm rate on the 2024 plume-free sets, two-stage mask IoU (end-to-end: an unflagged chip predicts an empty mask; and on flagged
chips only), and site-clustered 95 % bootstrap intervals. Also checks that the round-2 `fr_neg` numbers are reproduced.

**Limits (stated in the output too).** The split is temporal, not site-grouped: 799 of the 1,050 test plumes are at sites seen in
training. The val false-alarm pairs are 2021 scenes at any non-unseen-test site, the test ones are 2024 scenes at sites in no
training split, so the val FA rate need not transfer (round 2: 5 % on val gave 2.8 % on `real2024_strict`). 193 val plumes and 251
unseen-site test plumes make every rate uncertain by several points.

**How to run:** Runtime → Change runtime type → **T4 GPU** (CPU works, slower). No token needed: the weights and data are public.
*Runtime → Run all.* Results: `/content/methane-thresholds/out/thresholds.json` (send that file) and `config_thresholds.json`.

| step | measured: box CPU, 8 vCPU, fp32 (25 Nov 2026) | T4 (estimate) |
|---|---|---|
| fine-tune data: 6.2 GB zips → 8.9 GB cache | reused an existing cache | ~4–6 min (skipped if a cache exists) |
| plume-free pair caches: 2,000 val + 10,499 test pairs, HTTP range reads, 8.1 GB downloaded, 8.0 GB on disk | 3.5 min (60 pairs/s) | ~4–10 min (round 2 on Colab: ~21 pairs/s) |
| scoring 193 val + 1,050 test chips (each with its plume-free pair) and 12,499 pairs, fp32 | 7.4 min | ~2 min |
| **total** | **11.3 min** | **~10–20 min**, ~17 GB peak disk |

The box run reproduced the round-2 `fr_neg` numbers (val thresholds to 5 decimals, test IoU at 0.5 to 0.0001).

## 1. Config
The papermill `parameters` cell. `SMOKE_TEST = True` runs every cell on a few chips (CPU is fine); its numbers mean nothing.

In [ ]:
SMOKE_TEST = False          # True: tiny slices; proves the notebook runs. Numbers are meaningless.
USE_DRIVE = False           # True: keep the caches on Google Drive too (resume after a disconnect)
WEIGHTS = 'released'        # 'released': model.safetensors from HF_MODEL (public); or a local path to a .safetensors / .pt state dict
HF_MODEL = 'sighmon/methane-plume-se2-fullres'
CACHE_PATH = None           # optional existing 10 m fine-tune cache (cache10m.npy, 31 channels)
VAL_FA_N = 2000             # 2021 plume-free pairs for T_scene; 2000 with random_state 0 is the round-2 set
FA_N = None                 # None: every 2024 pair of the test FA sets (8,740 strict + 1,759 more at unseen-test sites)
VAL_FAS = [0.10, 0.05, 0.01]  # val false-alarm targets for T_scene
RECOMMENDED = '5%'          # which operating point config_thresholds.json marks as the default
PX_GRID = [round(0.05 * i, 2) for i in range(1, 20)]   # candidate pixel thresholds (0.05 ... 0.95)
PX_PICK_GRID = '20m'        # t_px is picked by val pixel IoU on this grid
AMP = False                 # fp32 by default: thresholds near 0.997 should be read at full precision, like the fp32 ONNX
BOOT_N = 1000               # site-clustered bootstrap resamples for the 95 % intervals
READ_THREADS = 128          # concurrent HTTP range reads
WORKERS = None              # DataLoader workers; None: min(4, cpu_count)
DISK_RESERVE_GB = 5
DELETE_ZIPS_AFTER_USE = True
WORK = None                 # None: /content/methane-thresholds on Colab, ./methane-thresholds elsewhere
FT_WORK = None              # where an earlier run left data/cache10m.npy; None: /content/methane-fullres on Colab

## 2. Setup

In [ ]:
import os, sys, subprocess, json, time, shutil, glob, importlib, importlib.util, hashlib
T0 = time.time(); TIMES = {}
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
PKGS = {'segmentation_models_pytorch': 'segmentation-models-pytorch==0.6.0', 'rasterio': 'rasterio', 'safetensors': 'safetensors',
        'psutil': 'psutil', 'huggingface_hub': 'huggingface_hub>=0.34', 'pyarrow': 'pyarrow', 'matplotlib': 'matplotlib', 'requests': 'requests'}
MISSING = [p for m, p in PKGS.items() if importlib.util.find_spec(m) is None]
if MISSING:
    print('installing missing packages:', MISSING); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *MISSING], check=True)
else: print('all packages already importable: skipping pip')
import numpy as np, pandas as pd, torch, psutil
if SMOKE_TEST: VAL_FA_N, FA_N, BOOT_N = 16, 24, 20
WORK = WORK or ('/content/methane-thresholds' if IN_COLAB else os.path.abspath('methane-thresholds'))
FT_WORK = FT_WORK or ('/content/methane-fullres' if IN_COLAB else os.path.abspath('methane-fullres'))
DATA, OUT = f'{WORK}/data', f'{WORK}/out'
for d in (DATA, OUT): os.makedirs(d, exist_ok=True)
DRIVE = None
if USE_DRIVE and IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/methane-thresholds'; os.makedirs(f'{DRIVE}/data', exist_ok=True)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKERS = WORKERS if WORKERS is not None else min(4, os.cpu_count() or 2)
GPU = torch.cuda.get_device_name(0) if DEV.type == 'cuda' else 'none (CPU)'
print(f'torch {torch.__version__} | device {DEV} | GPU {GPU} | cpus {os.cpu_count()} | RAM {psutil.virtual_memory().total / 1e9:.0f} GB '
      f'| free disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB | smoke {SMOKE_TEST}')
def drive_get(name):
    if DRIVE and not os.path.exists(f'{DATA}/{name}'):
        for f in glob.glob(f'{DRIVE}/data/{name}*'): print('restoring', os.path.basename(f), 'from Drive'); shutil.copy(f, DATA)
def drive_put(name):
    if DRIVE:
        for f in glob.glob(f'{DATA}/{name}*'):
            if not f.endswith('.tmp') and not os.path.exists(f'{DRIVE}/data/{os.path.basename(f)}'): shutil.copy(f, f'{DRIVE}/data/'); print('saved', os.path.basename(f), 'to Drive')
def gb(p): return os.path.getsize(p) / 1e9 if os.path.exists(p) else 0.0
def free_gb(): return shutil.disk_usage(WORK).free / 1e9
def need_disk(need, what):
    f = free_gb(); ok = f >= need + DISK_RESERVE_GB
    print(f'disk check, {what}: needs ~{need:.1f} GB + {DISK_RESERVE_GB} GB reserve, {f:.1f} GB free -> {"ok" if ok else "NOT ENOUGH"}'); return ok

## 3. Code
`methane_fullres.py` (model, pair building, pixel metrics) and `synplume.py` (range reads, pair caches, scene scores) as in round 2, and `thresholds.py` (the two-stage read-out and its metrics).

In [ ]:
%%writefile methane_fullres.py
"""Full-resolution (10 m, 200x200) methane plume detector for the Spiral Blue SE-2 (Jetson Orin NX).

Written by the Colab notebook methane_fullres_colab.ipynb. Same input contract as the CPU model
(sighmon/methane-plume-se2) except the spatial size:
    input  `reflectance`  float32 1x8x200x200  L1C DN/10000 at 10 m (2 km x 2 km chip)
           channels: target B2,B3,B4,B8A,B11,B12, then reference B11,B12
    output `plume_logits` float32 1x1x200x200  (sigmoid -> threshold)
The SWIR ratio channel is computed inside the graph, in fp32.

Cache layout (float16, N x 31 x 200 x 200, reflectance = DN/10000):
    0 plume mask, 1-6 target, 7-12 bg0, 13-18 bg1, 19-24 bg2, 25-30 bg3 (each B2,B3,B4,B8A,B11,B12).
    Runs without extra references read only channels 0-18 (mask, target, bg0, bg1): 39 % less I/O.
"""
import os, json, time, math, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

BANDS0 = [1, 2, 3, 8, 11, 12]  # 0-based in the 13-band L1C stack: B2,B3,B4,B8A,B11,B12
LEAVES = ['target', 'bg0', 'bg1', 'bg2', 'bg3']
NCH = 31; MASK = 0; NCH_BASIC = 19  # mask + target + bg0 + bg1
IN_CH = 8
INPUT_BANDS = ['T_B2', 'T_B3', 'T_B4', 'T_B8A', 'T_B11', 'T_B12', 'R_B11', 'R_B12']
META_COLS = ['id', 'split', 'detection:case_study', 'detection:ch4_fluxrate', 'site:country',
             'site:location_name', 'target:cloud', 'bg:cloud0', 'bg:cloud1', 'bg:cloud2', 'bg:cloud3',
             'bg:methane1', 'bg:methane2', 'bg:methane3', 'internal:source_file']

# ----------------------------------------------------------------------------------------- data prep
_P = {}
def _init(l1, zipdir):
    _P['l1'] = l1; _P['zip'] = zipdir
def _path(r):
    return f"/vsisubfile/{r['internal:offset']}_{r['internal:size']},{_P['zip']}/{r['internal:source_file']}"
def _work(args):
    import rasterio
    i, rows = args
    out = np.zeros((NCH, 200, 200), np.float16)
    for j, k in enumerate(LEAVES):
        with rasterio.open(_path(rows[k])) as d:
            out[1 + 6 * j:7 + 6 * j] = d.read([b + 1 for b in BANDS0]).astype(np.float32) / 10000.0
    with rasterio.open(_path(rows['plume'])) as d:
        out[MASK] = d.read(1).astype(np.float32)
    return i, out

def build_cache(l0, l1, zipdir, out_npy, workers=None):
    """Read 5 scenes + mask per sample straight out of the .tacozip files (GDAL /vsisubfile)."""
    from multiprocessing import Pool
    l1 = l1.copy(); l1['_pid'] = l1['internal:relative_path'].str.split('/').str[0]
    idx = {(p, k): j for j, (p, k) in enumerate(zip(l1['_pid'], l1['id']))}
    cols = ['internal:offset', 'internal:size', 'internal:source_file']
    jobs = [(i, {k: l1.loc[idx[(pid, k)], cols].to_dict() for k in LEAVES + ['plume']})
            for i, pid in enumerate(l0['id'])]
    X = np.lib.format.open_memmap(out_npy + '.tmp', 'w+', np.float16, (len(l0), NCH, 200, 200))
    t = time.time()
    with Pool(workers or os.cpu_count(), initializer=_init, initargs=(None, zipdir)) as p:
        for n, (i, a) in enumerate(p.imap_unordered(_work, jobs, chunksize=4)):
            X[i] = a
            if n % 250 == 0: print(f'  prep {n}/{len(jobs)}  {time.time() - t:.0f}s', flush=True)
    X.flush(); del X; os.replace(out_npy + '.tmp', out_npy)
    print(f'  prep done: {len(jobs)} chips in {time.time() - t:.0f}s')

def load_cache(npy, to_ram='auto'):
    X = np.load(npy, mmap_mode='r')
    if to_ram == 'auto':
        try:
            import psutil; to_ram = psutil.virtual_memory().available > 1.4 * X.nbytes
        except Exception: to_ram = False
    if to_ram: X = np.ascontiguousarray(X)
    print(f'cache {X.shape} {X.nbytes / 1e9:.1f} GB, in RAM: {bool(to_ram)}')
    return X

class Chips(torch.utils.data.Dataset):
    def __init__(self, X, idx, nch=NCH_BASIC): self.X, self.idx, self.nch = X, np.asarray(idx), nch
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = int(self.idx[i]); return torch.from_numpy(np.array(self.X[j, :self.nch])), j

def loader(X, idx, bs, shuffle, workers=2, seed=0, drop_last=False, nch=NCH_BASIC):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(Chips(X, idx, nch), batch_size=bs, shuffle=shuffle, drop_last=drop_last,
                                       num_workers=workers, pin_memory=torch.cuda.is_available(),
                                       persistent_workers=workers > 0, generator=g)

# ------------------------------------------------------------------------------------- batch building
def pair(a, t_blk, r_blk):
    """a: (B,C,H,W), C = 19 or 31. Block 0 = target, 1..4 = bg0..bg3. Build model input from per-sample
    target-block / reference-block indices (all 6 bands of the target block, B11/B12 of the reference)."""
    B = a.shape[0]; ar = torch.arange(B, device=a.device); nb = (a.shape[1] - 1) // 6
    blocks = a[:, 1:1 + 6 * nb].reshape(B, nb, 6, *a.shape[2:])
    return torch.cat([blocks[ar, t_blk], blocks[ar, r_blk][:, 4:6]], 1)

def eval_batch(a):
    """Fixed eval pairs, same as the CPU model: plume pair target vs bg0; plume-free pair bg1 vs bg0."""
    a = a.float(); B = a.shape[0]
    z = torch.zeros(B, dtype=torch.long, device=a.device)
    x = pair(a, z, z + 1); xn = pair(a, z + 2, z + 1)
    y = (a[:, MASK:MASK + 1] >= 0.5).float()
    return x, y, xn

def train_batch(a, rows, gen, cfg, meth):
    """Training pairs. cfg keys: neg_p, extra_refs, extra_ref_p, neg_max_methane, aug.
    meth: (N,3) tensor bg:methane1..3 on device (fraction of reference pixels with own enhancement)."""
    a = a.float(); B, dev = a.shape[0], a.device
    r = lambda *s: torch.rand(*s, generator=gen, device=dev)
    ri = lambda hi, *s: torch.randint(0, hi, s, generator=gen, device=dev)
    t_blk = torch.zeros(B, dtype=torch.long, device=dev); r_blk = torch.ones_like(t_blk)  # target vs bg0
    y = (a[:, MASK:MASK + 1] >= 0.5).float()
    if cfg.get('extra_refs'):  # swap the reference for bg1..bg3 (block 2..4)
        sw = r(B) < cfg.get('extra_ref_p', 0.5)
        r_blk = torch.where(sw, 2 + ri(3, B), r_blk)
    if cfg.get('neg_p', 0) > 0:  # plume-free pair: one reference used as "target" against another
        neg = r(B) < cfg['neg_p']
        if cfg.get('extra_refs'):
            nt = 2 + ri(3, B)                                   # bg1..bg3 as target
            nr = (nt - 1 + 1 + ri(3, B)) % 4 + 1                # any other of bg0..bg3
        else:
            nt = torch.full_like(t_blk, 2); nr = torch.ones_like(t_blk)  # bg1 vs bg0 (CPU recipe)
        thr = cfg.get('neg_max_methane')
        if thr is not None:  # skip "negatives" whose target reference has its own enhancement
            neg &= meth[rows, (nt - 2)] <= thr
        t_blk = torch.where(neg, nt, t_blk); r_blk = torch.where(neg, nr, r_blk)
        y = torch.where(neg[:, None, None, None], torch.zeros_like(y), y)
    x = pair(a, t_blk, r_blk)
    if cfg.get('aug', 0) >= 1:  # per-sample rot90 + horizontal flip
        k = ri(4, B).tolist(); f = (r(B) < 0.5).tolist()  # one host sync per batch
        xs, ys = [], []
        for i in range(B):
            xi, yi = torch.rot90(x[i], k[i], (1, 2)), torch.rot90(y[i], k[i], (1, 2))
            if f[i]: xi, yi = xi.flip(-1), yi.flip(-1)
            xs.append(xi); ys.append(yi)
        x, y = torch.stack(xs), torch.stack(ys)
    if cfg.get('aug', 0) >= 2:  # radiometric jitter: per-chip gain on all bands (scene brightness)
        x = x * (0.9 + 0.2 * r(B, 1, 1, 1))
    return x.contiguous(), y.contiguous()

# ------------------------------------------------------------------------------------------- model
class MethaneNet(nn.Module):
    """Raw TOA reflectance (B,8,H,W) -> plume logits (B,1,H,W). Reflect-pads to a multiple of 32."""
    def __init__(self, encoder='mobilenet_v2', weights='imagenet', decoder=(128, 64, 32, 16, 8)):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.norm = nn.BatchNorm2d(IN_CH + 1)
        self.net = smp.Unet(encoder, encoder_weights=weights, in_channels=IN_CH + 1, classes=1,
                            decoder_channels=tuple(decoder))
        if hasattr(self.net.encoder, 'set_swish'):  # efficientnet-pytorch: ONNX-exportable swish
            self.net.encoder.set_swish(memory_efficient=False)
    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):  # ratio head in fp32 always
            x = x.float()
            t11, t12, r11, r12 = x[:, 4:5], x[:, 5:6], x[:, 6:7], x[:, 7:8]
            eps = 1e-3
            ratio = torch.log((t12 + eps) / (t11 + eps)) - torch.log((r12 + eps) / (r11 + eps))
            ratio = (ratio - ratio.mean(dim=(2, 3), keepdim=True)) * 20.0
            z = self.norm(torch.cat([x, ratio], 1))
        H, W = int(x.shape[2]), int(x.shape[3])
        ph, pw = (-H) % 32, (-W) % 32
        pt, pl = ph // 2, pw // 2
        if ph or pw: z = F.pad(z, (pl, pw - pl, pt, ph - pt), mode='reflect')
        y = self.net(z)
        return y[:, :, pt:pt + H, pl:pl + W]

def amp_dtype_for(dev):
    """bf16 on Ampere+ (A100, L4); fp16 + GradScaler on T4/V100 (bf16 there is emulated and slow)."""
    if dev.type != 'cuda': return None
    return torch.bfloat16 if torch.cuda.get_device_capability(dev)[0] >= 8 else torch.float16

def dice_bce(logits, y):
    logits = logits.float()
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=torch.tensor(3.0, device=y.device))
    p = torch.sigmoid(logits); inter = (p * y).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + y.sum((1, 2, 3))
    return bce + (1 - (2 * inter + 1) / (den + 1)).mean()

# ------------------------------------------------------------------------------------------ metrics
THRS = np.round(np.arange(0.05, 0.951, 0.05), 2)

@torch.no_grad()
def predict_counts(model, dl, dev, amp_dtype=None, thrs=None):
    """One pass over a split. For every threshold in THRS and for two grids
    ('10m' native, min 40 px = 0.4 ha; '20m' = 2x2 mean-pooled probabilities vs the CPU model's 20 m
    labels (pooled fraction >= 0.5), min 10 px = 0.4 ha) accumulate pixel tp/fp/fn, chip detections on
    plume pairs and false alarms on plume-free pairs (bg1 vs bg0). thrs: optional threshold list (default THRS)."""
    thrs = THRS if thrs is None else np.asarray(sorted(set(float(t) for t in thrs)), dtype=np.float64)
    model.eval(); T = torch.tensor(thrs, device=dev, dtype=torch.float32)[:, None, None, None, None]
    acc = {g: dict(tp=torch.zeros(len(thrs), device=dev, dtype=torch.float64), fp=None, fn=None, det=None, fa=None)
           for g in ('10m', '20m')}
    for g in acc:
        for k in ('fp', 'fn', 'det', 'fa'): acc[g][k] = torch.zeros_like(acc[g]['tp'])
    n = 0
    for a, _ in dl:
        a = a.to(dev, non_blocking=True); x, y, xn = eval_batch(a)
        with torch.autocast(dev.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            p = torch.sigmoid(model(x).float()); pn = torch.sigmoid(model(xn).float())
        for g, (pp, yy, ppn, mpx) in {
                '10m': (p, y, pn, 40),
                '20m': (F.avg_pool2d(p, 2), (F.avg_pool2d(y, 2) >= 0.5).float(), F.avg_pool2d(pn, 2), 10)}.items():
            b = pp[None] >= T; t = yy[None].bool(); bn = ppn[None] >= T
            A = acc[g]
            A['tp'] += (b & t).sum((1, 2, 3, 4)); A['fp'] += (b & ~t).sum((1, 2, 3, 4)); A['fn'] += (~b & t).sum((1, 2, 3, 4))
            flagged = b.flatten(2).sum(2) >= mpx
            A['det'] += (flagged & (b & t).flatten(2).any(2)).sum(1)
            A['fa'] += (bn.flatten(2).sum(2) >= mpx).sum(1)
        n += a.shape[0]
    out = {}
    for g, A in acc.items():
        A = {k: v.cpu().numpy() for k, v in A.items()}
        P = A['tp'] / np.maximum(A['tp'] + A['fp'], 1); R = A['tp'] / np.maximum(A['tp'] + A['fn'], 1)
        out[g] = dict(iou=A['tp'] / np.maximum(A['tp'] + A['fp'] + A['fn'], 1), f1=2 * P * R / np.maximum(P + R, 1e-9),
                      precision=P, recall=R, det_rate=A['det'] / max(n, 1), false_alarm=A['fa'] / max(n, 1))
    if thrs is not THRS: out['thrs'] = thrs
    return out

def at(counts, thr, grid='10m'):
    t = counts.get('thrs', THRS); i = int(np.argmin(np.abs(t - thr))); return {k: float(v[i]) for k, v in counts[grid].items()}

def pick_threshold(val_counts, metric='f1', grid='10m'):
    return float(THRS[int(np.argmax(val_counts[grid][metric]))])

# ------------------------------------------------------------------------------------------ training
def train_run(name, X, meta, dev, out_root, *, encoder='mobilenet_v2', init=None, epochs=15, lr=1e-3, bs=16,
              seed=0, aug=1, neg_p=0.0, extra_refs=False, extra_ref_p=0.5, neg_max_methane=None, amp=True,
              workers=2, project='methane-se2-fullres', preview_every=3, eval_every=1, log=print,
              skip_done=True, channels_last=True):
    import trackio
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    out = f'{out_root}/{name}'; os.makedirs(out, exist_ok=True)
    if skip_done and os.path.exists(f'{out}/final.pt'):
        log(f'[{name}] final.pt exists, skipping (set SKIP_DONE=False to retrain)'); return f'{out}/final.pt'
    cfg = dict(name=name, encoder=encoder, init=init, epochs=epochs, lr=lr, bs=bs, seed=seed, aug=aug, neg_p=neg_p,
               extra_refs=extra_refs, extra_ref_p=extra_ref_p, neg_max_methane=neg_max_methane, amp=amp,
               resolution='10m', chip=200)
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    tr = np.where(meta.split == 'train')[0]; va = np.where(meta.split == 'validation')[0]
    meth = torch.tensor(meta[['bg:methane1', 'bg:methane2', 'bg:methane3']].fillna(1.0).values, device=dev, dtype=torch.float32)
    model = MethaneNet(encoder, weights=None if init else 'imagenet')
    if init: model.load_state_dict(torch.load(init, map_location='cpu'))
    model.to(dev)
    if channels_last and dev.type == 'cuda': model = model.to(memory_format=torch.channels_last)
    amp_dtype = None
    if amp and dev.type == 'cuda':
        amp_dtype = amp_dtype_for(dev)
    scaler = torch.amp.GradScaler('cuda', enabled=amp_dtype == torch.float16)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    dl = loader(X, tr, bs, True, workers, seed, drop_last=True, nch=NCH if extra_refs else NCH_BASIC)
    dv = loader(X, va, 32, False, workers)
    spe = max(len(dl), 1); steps = epochs * spe; warm = 0 if init else spe
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) if warm and s < warm else
                                              0.5 * (1 + math.cos(math.pi * min(1, (s - warm) / max(1, steps - warm)))))
    trackio.init(project=project, name=name, config=cfg)
    va_vis = va[:4]
    def preview(ep):
        model.eval(); a = torch.from_numpy(np.array(X[np.sort(va_vis), :NCH_BASIC])).to(dev); x, y, _ = eval_batch(a)
        with torch.no_grad(): p = torch.sigmoid(model(x).float()).cpu()
        x = x.cpu(); f, ax = plt.subplots(3, len(va_vis), figsize=(2.2 * len(va_vis), 6.6), squeeze=False)
        for i in range(len(va_vis)):
            ax[0, i].imshow(np.clip(x[i, [2, 1, 0]].permute(1, 2, 0).numpy() * 3.5, 0, 1))
            ax[1, i].imshow(y[i, 0].cpu(), cmap='gray'); ax[2, i].imshow(p[i, 0], cmap='magma', vmin=0, vmax=1)
        for r in ax.ravel(): r.axis('off')
        f.suptitle(f'{name} epoch {ep}: RGB / label / prediction (10 m)', fontsize=9); f.tight_layout()
        fn = f'{out}/preview_ep{ep:03d}.png'; f.savefig(fn, dpi=70); plt.close(f); return fn
    hist = []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); ls = []
        for a, rows in dl:
            a = a.to(dev, non_blocking=True); rows = rows.to(dev)
            x, y = train_batch(a, rows, gen, cfg, meth)
            if channels_last and dev.type == 'cuda': x = x.to(memory_format=torch.channels_last)
            with torch.autocast(dev.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                loss = dice_bce(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            ls.append(loss.item())
        rec = dict(epoch=ep, train_loss=float(np.mean(ls)) if ls else float('nan'), lr=opt.param_groups[0]['lr'],
                   epoch_s=time.time() - t0)
        if ep % eval_every == 0 or ep == epochs:
            c = predict_counts(model, dv, dev, amp_dtype)
            for g in ('10m', '20m'):
                rec.update({f'val{"" if g == "10m" else "20"}/{k}': v for k, v in at(c, 0.5, g).items()})
            rec['val/best_f1_thr'] = pick_threshold(c)
        lg = dict(rec)
        if ep % preview_every == 0 or ep == epochs: lg['val/preview'] = trackio.Image(preview(ep))
        trackio.log(lg); hist.append(rec)
        log(f'[{name}] ep {ep}/{epochs} loss {rec["train_loss"]:.4f} val IoU {rec.get("val/iou", float("nan")):.3f} '
            f'F1 {rec.get("val/f1", float("nan")):.3f} FA {rec.get("val/false_alarm", float("nan")):.3f} ({rec["epoch_s"]:.0f}s)')
    sd = {k: v.contiguous() for k, v in model.state_dict().items()}
    torch.save(sd, f'{out}/final.pt.tmp'); os.replace(f'{out}/final.pt.tmp', f'{out}/final.pt')
    json.dump(dict(config=cfg, history=hist), open(f'{out}/history.json', 'w'), indent=1)
    trackio.finish()
    return f'{out}/final.pt'

def average(states):
    out = {}
    for k in states[0]:
        v = [s[k] for s in states]
        out[k] = (sum(t.float() for t in v) / len(v)).to(v[0].dtype) if v[0].is_floating_point() else v[0]
    return out

def evaluate_state(state, encoder, X, idx, dev, workers=2, amp=True):
    m = MethaneNet(encoder, weights=None); m.load_state_dict(state); m.to(dev)
    amp_dtype = amp_dtype_for(dev) if amp else None
    return predict_counts(m, loader(X, idx, 32, False, workers), dev, amp_dtype)

In [ ]:
%%writefile synplume.py
"""Synthetic-plume training for the full-res methane detector (colab/PLAN_synthetic_plumes.md).

Written by synthetic_plumes_colab.ipynb next to methane_fullres.py, which it imports as `mf` (model, pair building,
pixel metrics). Nothing here changes the onboard input contract: `reflectance` float32 1x8x200x200 (L1C DN/10000,
10 m; target B2,B3,B4,B8A,B11,B12 then reference B11,B12) -> `plume_logits` 1x1x200x200.

Pieces
  * LeafReader: HTTP range reads of single GeoTIFF leaves out of the MethaneSET .tacozip files on the HF hub
    (or from a local copy of the zip), so the 36.9 GB pretraining set is never downloaded whole.
  * pair caches: plume-free pretraining pairs as the 8 model channels, float16, memory-mapped, resumable.
  * bank cache: WRF-LES XCH4 enhancements (ppb at 3000 kg/h, 20 m, wind -> +x) on a fixed canvas.
  * Transmittance: MethaneSET integrated_transmittances.json -> relative B11/B12 transmittance T(1800+d)/T(1800).
  * inject(): GPU, per batch. Rotate the plume to the scene wind, place it, scale to Q, multiply target B11/B12.
  * MethaneNetMBMP: MethaneNet plus a Varon MBMP channel computed in-graph (fp32); same 8-channel input.
  * train_syn(), scene scores, PR / AP and operating points at fixed real false-alarm rates.
"""
import os, io, json, time, math, threading, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import methane_fullres as mf

HF_DS = 'https://huggingface.co/datasets/tacofoundation/methaneset/resolve/main'
BANDS0 = mf.BANDS0                       # B2,B3,B4,B8A,B11,B12 (0-based in the 13-band stack)
CANVAS_H, CANVAS_W, CAN_R, CAN_C = 160, 192, 80.0, 32.0   # bank canvas at 20 m; source pixel centre at (row 80, col 32)
SENSORS = ['S2A', 'S2B', 'S2C']
BG_PPB = 1800.0

# ----------------------------------------------------------------------------------------- leaf reads
class LeafReader:
    """Read byte ranges (one GeoTIFF leaf) from tacozip files: local file if present, else HTTP range request.
    The signed CDN URL behind each HF `resolve` link is looked up once per file and refreshed on 403.
    HF auth headers (if any token is configured) are sent to huggingface.co only and never printed."""
    def __init__(self, local_dirs=()):
        self.local_dirs = [d for d in local_dirs if d]; self.loc = {}; self.tl = threading.local(); self.lock = threading.Lock()
        try:
            from huggingface_hub.utils import build_hf_headers; self.h = build_hf_headers()
        except Exception: self.h = {}
        self.bytes = 0; self.remote = 0; self.retries = {}
    def _s(self):
        import requests
        if not hasattr(self.tl, 's'): self.tl.s = requests.Session()
        return self.tl.s
    def _local(self, sub, f):
        for d in self.local_dirs:
            for p in (f'{d}/{sub}/{f}', f'{d}/{f}'):
                if os.path.exists(p): return p
        return None
    def _url(self, sub, f, refresh=False):
        k = (sub, f)
        if refresh or k not in self.loc:
            r = self._s().head(f'{HF_DS}/{sub}/{f}', headers=self.h, allow_redirects=False, timeout=60)
            u = r.headers.get('location') or f'{HF_DS}/{sub}/{f}'
            with self.lock: self.loc[k] = u
        return self.loc[k]
    def read(self, sub, f, off, size):
        if not isinstance(f, str) or not f.endswith('.tacozip'):
            raise TypeError(f'leaf file name must be a .tacozip str, got {type(f).__name__}: {str(f)[:80]!r}')
        off, size = int(off), int(size)
        p = self._local(sub, f)
        if p:
            with open(p, 'rb') as fh: fh.seek(off); b = fh.read(size)
        else:
            for attempt in range(6):
                try:
                    r = self._s().get(self._url(sub, f, refresh=attempt > 0), headers={'Range': f'bytes={off}-{off + size - 1}'}, timeout=120)
                    if r.status_code == 206 and len(r.content) == size: b = r.content; break
                    why = f'HTTP {r.status_code}'
                except Exception as e: why = type(e).__name__
                with self.lock: self.retries[why] = self.retries.get(why, 0) + 1
                time.sleep(min(30, 2 ** attempt) * (0.5 + np.random.rand()))
            else: raise IOError(f'range read failed: {sub}/{f} @{off}+{size}')
            self.remote += 1
        self.bytes += len(b); return b

def decode(b, bands=None):
    import rasterio
    from rasterio.io import MemoryFile
    with MemoryFile(b) as m, m.open() as d:
        return d.read([i + 1 for i in bands] if bands is not None else None).astype(np.float32)

def leaf_index(l1):
    """(folder id, leaf id) -> (offset, size, source_file) as plain Python int/int/str, in a dict.
    Built once, single-threaded. Worker threads only read the dict. (Concurrent DataFrame.loc lookups on an unsorted
    MultiIndex are not thread-safe in pandas 2.2, the Colab version: a few of them return a one-row DataFrame, so the
    "file name" became a pandas Series and the range read failed.) Duplicate keys are counted: identical duplicates
    keep the first row, conflicting duplicates raise."""
    pid = l1['internal:relative_path'].str.split('/').str[0].values
    lid = l1['id'].astype(str).values
    off = l1['internal:offset'].astype('int64').values; size = l1['internal:size'].astype('int64').values
    src = l1['internal:source_file'].astype(str).values
    out, ndup, conflict = {}, 0, []
    for k in zip(pid, lid, off, size, src):
        key, val = (str(k[0]), str(k[1])), (int(k[2]), int(k[3]), str(k[4]))
        if key in out:
            ndup += 1
            if out[key] != val: conflict.append((key, out[key], val))
            continue
        out[key] = val
    if ndup: print(f'leaf_index: {ndup} duplicate (folder, leaf) rows, {len(conflict)} conflicting')
    if conflict: raise ValueError(f'conflicting duplicate leaves, e.g. {conflict[:3]}')
    return out

def _parallel(fn, n, workers, label, every=500):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    t = time.time(); done = 0
    with ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(fn, i) for i in range(n)]
        for f in as_completed(futs):
            f.result(); done += 1
            if done % every == 0 or done == n: print(f'  {label} {done}/{n}  {time.time() - t:.0f}s', flush=True)
    return time.time() - t

def build_pair_cache(rows, L1, sub, out_npy, reader, workers=64):
    """Plume-free pairs -> (N,8,200,200) float16 [target B2,B3,B4,B8A,B11,B12, reference B11,B12], DN/10000.
    Resumable: a sidecar .done.npy marks finished rows; rows that fail to read are marked in .bad.npy (excluded later)."""
    N = len(rows); tmp = out_npy + '.tmp'; dn = out_npy + '.done.npy'; bad_p = out_npy + '.bad.npy'
    if os.path.exists(out_npy):
        return np.load(out_npy, mmap_mode='r'), (np.load(bad_p) if os.path.exists(bad_p) else np.zeros(N, bool))
    X = np.lib.format.open_memmap(tmp, 'r+' if os.path.exists(tmp) and os.path.exists(dn) else 'w+', np.float16, (N, 8, 200, 200))
    done = np.load(dn) if os.path.exists(dn) else np.zeros(N, bool)
    bad = np.load(bad_p) if os.path.exists(bad_p) else np.zeros(N, bool)
    todo = np.where(~done)[0]; ids = [str(v) for v in rows['id'].values]; lock = threading.Lock(); cnt = [0]
    print(f'pair cache {out_npy}: {N} pairs, {len(todo)} to fetch')
    def work(k):
        i = int(todo[k])
        try:
            to, ts, tf = L1[(ids[i], 'target')]; ro, rs, rf = L1[(ids[i], 'reference')]
            a = decode(reader.read(sub, tf, to, ts), BANDS0)
            b = decode(reader.read(sub, rf, ro, rs), BANDS0[4:6])
            assert a.shape == (6, 200, 200) and b.shape == (2, 200, 200)
            X[i] = np.concatenate([a, b]) / 10000.0
        except Exception as e:
            bad[i] = True; print('  bad pair', i, type(e).__name__, str(e)[:120])
        with lock:
            done[i] = True; cnt[0] += 1
            if cnt[0] % 1000 == 0: X.flush(); np.save(dn, done); np.save(bad_p, bad)
    s = _parallel(work, len(todo), workers, 'pairs')
    X.flush(); del X; np.save(bad_p, bad); os.replace(tmp, out_npy)
    if os.path.exists(dn): os.remove(dn)
    print(f'  pair cache done in {s:.0f}s ({len(todo) / max(s, 1e-9):.1f} pairs/s), {bad.sum()} bad, '
          f'{os.path.getsize(out_npy) / 1e9:.2f} GB on disk')
    return np.load(out_npy, mmap_mode='r'), bad

def build_leaf_cache(l0, L1, leaf, sub, out_npy, reader, workers=32, band=1):
    """One single-band leaf per fine-tune sample (e.g. the MBMP `ch4` leaf) -> (N,200,200) float32.
    Resumable like the pair caches: a .done.npy sidecar marks finished rows. A partial .tmp without a sidecar (left by
    an older version) is discarded and rebuilt. Every key is checked before any read starts."""
    if os.path.exists(out_npy): return np.load(out_npy, mmap_mode='r')
    N = len(l0); ids = [str(v) for v in l0['id'].values]; tmp = out_npy + '.tmp'; dn = out_npy + '.done.npy'
    missing = [i for i in ids if (i, leaf) not in L1]
    if missing: raise KeyError(f'{len(missing)} samples have no {leaf!r} leaf, e.g. {missing[:3]}')
    resume = os.path.exists(tmp) and os.path.exists(dn)
    W = np.lib.format.open_memmap(tmp, 'r+' if resume else 'w+', np.float32, (N, 200, 200))
    done = np.load(dn) if resume else np.zeros(N, bool); todo = np.where(~done)[0]; lock = threading.Lock(); cnt = [0]
    print(f'{leaf} cache {out_npy}: {N} samples, {len(todo)} to read' + (' (resuming)' if resume else ''))
    def work(k):
        i = int(todo[k]); o, sz, f = L1[(ids[i], leaf)]
        W[i] = decode(reader.read(sub, f, o, sz), [band - 1])[0]
        with lock:
            done[i] = True; cnt[0] += 1
            if cnt[0] % 500 == 0: W.flush(); np.save(dn, done)
    try:
        s = _parallel(work, len(todo), workers, leaf, 1000)
    finally:
        W.flush(); np.save(dn, done)
    del W; os.replace(tmp, out_npy); os.remove(dn)
    print(f'  {leaf} cache done in {s:.0f}s'); return np.load(out_npy, mmap_mode='r')

# ---------------------------------------------------------------------------------------------- bank
def assign_bank(scenes, bank, k, seed=0, sza_max=71.03):
    """For each scene pick k random physical plumes and, for each, the bank row with the nearest sun position
    (distance between solar-parallax vectors tan(SZA)*(cos RAA, sin RAA), the bank's own 60 m spacing).
    Scenes without a RAA (calm wind) get a random RAA. Returns (n_scenes, k) bank row numbers."""
    rng = np.random.default_rng(seed)
    uids = bank['methane:plume_uid'].values
    first = bank[uids == uids[0]]
    sun = np.stack([np.tan(np.radians(first['sun:sza'].values)) * np.cos(np.radians(first['sun:raa'].values)),
                    np.tan(np.radians(first['sun:sza'].values)) * np.sin(np.radians(first['sun:raa'].values))], 1)
    sun_key = list(zip(first['sun:sza'].round(4), first['sun:raa'].round(4)))
    pos = {(u, s): i for i, (u, s) in enumerate(zip(uids, zip(bank['sun:sza'].round(4), bank['sun:raa'].round(4))))}
    plumes = np.unique(uids)
    sza = np.clip(scenes['target:sza'].values, 0, sza_max)
    raa = scenes['target:raa'].values.copy(); nr = ~np.isfinite(raa); raa[nr] = rng.uniform(0, 360, nr.sum())
    v = np.stack([np.tan(np.radians(sza)) * np.cos(np.radians(raa)), np.tan(np.radians(sza)) * np.sin(np.radians(raa))], 1)
    near = np.argmin(((v[:, None, :] - sun[None]) ** 2).sum(-1), 1)
    out = np.zeros((len(scenes), k), np.int64)
    for i in range(len(scenes)):
        for j, u in enumerate(rng.choice(plumes, k, replace=False)):
            out[i, j] = pos[(u, sun_key[near[i]])]
    return out

def build_bank_cache(bank_rows, out_npy, reader, sub='methaneset-bank', workers=64):
    """Bank rows -> (M,160,192) float16 canvas at 20 m, ppb at 3000 kg/h, wind -> +x (columns), +y up (row 0 = top).
    Each crop is pasted with an integer offset so that its source pixel centre lands at (CAN_R + fr, CAN_C + fc),
    fr, fc in [0, 1); the exact canvas source position is returned. The canvas keeps 3.2 km downwind, 1.6 km crosswind
    each side and 640 m upwind of the source; the fraction of plume mass that falls outside is returned as `lost`."""
    meta_p = out_npy.replace('.npy', '_meta.npz')
    if os.path.exists(out_npy) and os.path.exists(meta_p):
        z = np.load(meta_p); return np.load(out_npy, mmap_mode='r'), z['src'], z['lost']
    M = len(bank_rows); C = np.lib.format.open_memmap(out_npy + '.tmp', 'w+', np.float16, (M, CANVAS_H, CANVAS_W))
    src = np.zeros((M, 2), np.float32); lost = np.zeros(M, np.float32)
    R = [dict(f=str(a), o=int(b), s=int(c), sr=float(d), sc=float(e)) for a, b, c, d, e in zip(  # plain values: no pandas in threads
        bank_rows['internal:source_file'], bank_rows['internal:offset'], bank_rows['internal:size'], bank_rows['array:source_row'], bank_rows['array:source_col'])]
    def work(i):
        r = R[i]
        a = decode(reader.read(sub, r['f'], r['o'], r['s']))[0]
        a = np.nan_to_num(a, nan=0.0)
        sr, sc = r['sr'], r['sc']
        r0, c0 = int(np.floor(CAN_R - sr)), int(np.floor(CAN_C - sc))
        src[i] = (sr + r0, sc + c0)
        H, W = a.shape; rr0, cc0 = max(r0, 0), max(c0, 0); rr1, cc1 = min(r0 + H, CANVAS_H), min(c0 + W, CANVAS_W)
        if rr1 > rr0 and cc1 > cc0:
            C[i, rr0:rr1, cc0:cc1] = a[rr0 - r0:rr1 - r0, cc0 - c0:cc1 - c0]
        tot = float(a.sum()); lost[i] = max(0.0, 1 - float(C[i].astype(np.float32).sum()) / tot) if tot > 0 else 0.0
    s = _parallel(work, M, workers, 'bank', 2000); C.flush(); del C; os.replace(out_npy + '.tmp', out_npy)
    np.savez(meta_p, src=src, lost=lost)
    print(f'  bank cache: {M} plumes in {s:.0f}s, mass outside canvas median {np.median(lost):.3%} '
          f'p95 {np.percentile(lost, 95):.3%}, {os.path.getsize(out_npy) / 1e9:.2f} GB')
    return np.load(out_npy, mmap_mode='r'), src, lost

# ------------------------------------------------------------------------------------- transmittance
class Transmittance:
    """MethaneSET integrated_transmittances.json. Per sensor, tables over 8 AMFs x 9 CH4 mixing ratios.
    Units: B11 is a plain transmittance (~0.99) but B12 is tabulated on another scale (~64; its `_bg` column is the
    constant 64.2987 = the B12 value at 1800 ppb for every AMF). Only the ratio T(1800 + d) / T(1800) at the same
    AMF enters the injection, so the scale cancels; check_units() verifies that the 1800 ppb interpolation equals
    the tabulated background for both bands at every AMF before anything is trusted.
    Beyond the last mixing ratio (~8,280 ppb), ln T is extrapolated linearly from the last segment."""
    def __init__(self, path):
        d = json.load(open(path)); self.raw = d
        self.amf = torch.tensor(d['amf_arr'], dtype=torch.float32)                      # (8,)
        self.mr = torch.tensor(d['mr_ch4_arr'], dtype=torch.float32)                    # (8,9)
        self.bg = float(d['background_concentration']); assert abs(self.bg - BG_PPB) < 1e-6
        self.T = {b: torch.stack([torch.tensor(d[s][f'transmittance_{b}'], dtype=torch.float32) for s in SENSORS])
                  for b in ('b11', 'b12')}                                               # (3,8,9)
        self.Tbg = {b: torch.stack([torch.tensor(np.ravel(d[s][f'transmittance_{b}_bg']), dtype=torch.float32)
                                    for s in SENSORS]) for b in ('b11', 'b12')}          # (3,8)
        self.with_ltoa = d.get('with_ltoa_correction')
        self._cache = {}
    def to(self, dev):
        """Kept for older notebooks. The master tables stay on the CPU; every call uses a copy on the device of its
        inputs (see _on), so the same object serves CPU and GPU callers."""
        return self
    def _on(self, dev):
        """(amf, mr, T) on `dev`, copied once per device and cached."""
        k = str(dev)
        if k not in self._cache:
            self._cache[k] = (self.amf.to(dev), self.mr.to(dev), {b: v.to(dev) for b, v in self.T.items()})
        return self._cache[k]
    @staticmethod
    def _interp1(x, xp, fp, log_extrap=True):
        """Batched 1-D interpolation. x (B,N), xp (B,K) increasing, fp (B,K). Linear in ln f, linear extrapolation."""
        K = xp.shape[1]; i = torch.searchsorted(xp.contiguous(), x.contiguous()).clamp(1, K - 1)
        x0, x1 = xp.gather(1, i - 1), xp.gather(1, i); lf = torch.log(fp)
        y0, y1 = lf.gather(1, i - 1), lf.gather(1, i)
        return torch.exp(y0 + (y1 - y0) * (x - x0) / (x1 - x0))
    def rows_at(self, sensor, amf):
        """Per-chip table rows at the chip's AMF (linear in AMF). sensor (B,) long, amf (B,). -> mr (B,9), T{b}: (B,9)."""
        A, MR, TT = self._on(amf.device); sensor = sensor.to(amf.device).long()
        a = amf.float().clamp(float(self.amf[0]), float(self.amf[-1])); K = len(self.amf)
        j = torch.searchsorted(A.contiguous(), a.contiguous()).clamp(1, K - 1)
        w = ((a - A[j - 1]) / (A[j] - A[j - 1]))[:, None]
        mr = MR[j - 1] * (1 - w) + MR[j] * w
        T = {b: TT[b][sensor, j - 1] * (1 - w) + TT[b][sensor, j] * w for b in TT}
        return mr, T
    def rel(self, sensor, amf, delta):
        """T(1800 + delta) / T(1800) for B11 and B12. delta (B,N) ppb >= 0. -> (rel11, rel12) each (B,N)."""
        amf = amf.to(delta.device); mr, T = self.rows_at(sensor, amf); B = delta.shape[0]
        out = []
        for b in ('b11', 'b12'):
            t = self._interp1(BG_PPB + delta, mr, T[b]); t0 = self._interp1(torch.full((B, 1), BG_PPB, device=delta.device), mr, T[b])
            out.append(t / t0)
        return out
    def check_units(self, tol=0.005):
        """Interpolated T(1800) vs the tabulated *_bg value, every sensor/AMF/band. Returns (max rel. error, report dict)."""
        rep = {}; worst = 0.0
        for si, s in enumerate(SENSORS):
            for b in ('b11', 'b12'):
                t0 = self._interp1(torch.full((len(self.amf), 1), BG_PPB), self.mr.cpu(), self.T[b][si].cpu())[:, 0]
                e = float(((t0 - self.Tbg[b][si].cpu()) / self.Tbg[b][si].cpu()).abs().max()); worst = max(worst, e)
                rep[f'{s}_{b}'] = dict(T_at_1800=[round(float(v), 5) for v in t0], tabulated_bg=[round(float(v), 5) for v in self.Tbg[b][si]], max_rel_err=e)
        return worst, rep
    def sensitivity(self, sensor, amf, d=1000.0):
        """d ln(B12/B11) per ppb at the chip AMF, from a +1000 ppb step (negative)."""
        B = amf.shape[0]; r11, r12 = self.rel(sensor, amf, torch.full((B, 1), d, device=amf.device))
        return (torch.log(r12) - torch.log(r11))[:, 0] / d

# ------------------------------------------------------------------------------------- SWIR features
def log_ratio(x):
    """Same as the model's fp32 head (before mean removal): ln(T12/T11) - ln(R12/R11). x (B,8,H,W)."""
    eps = 1e-3; x = x.float()
    return torch.log((x[:, 5:6] + eps) / (x[:, 4:5] + eps)) - torch.log((x[:, 7:8] + eps) / (x[:, 6:7] + eps))

def mbmp(x):
    """Varon-style multi-band multi-pass: (c*B12 - B11)/B11 per image, c fitted per chip by least squares
    (c = sum(B11*B12)/sum(B12^2)), target minus reference. Negative where methane absorbs B12."""
    x = x.float(); eps = 1e-6
    def mb(b11, b12):
        c = (b11 * b12).sum((2, 3), keepdim=True) / ((b12 * b12).sum((2, 3), keepdim=True) + eps)
        return (c * b12 - b11) / (b11 + 1e-3)
    return mb(x[:, 4:5], x[:, 5:6]) - mb(x[:, 6:7], x[:, 7:8])

def robust_sigma(z):
    """1.4826 * MAD over each chip. z (B,1,H,W) -> (B,)."""
    f = z.flatten(1); m = f.median(1, keepdim=True).values
    return 1.4826 * (f - m).abs().median(1).values

# ---------------------------------------------------------------------------------------- injection
def inject(x, canv, src, theta, sza, vza, sensor, edge_ppb, tm, gen, *, q_min=500., q_max=20000., q_fixed=None,
           floor_sigma=0.5, centre_jitter_px=60, wind_jitter_deg=15.0):
    """x (B,8,200,200) clean pair on device, float. canv (B,1,160,192) ppb at 3000 kg/h. src (B,2) canvas source (row,col).
    theta (B,) wind direction (radians, counter-clockwise from east, direction the wind blows towards; NaN = random).
    Returns x_inj, label (B,1,H,W), delta (B,1,H,W) ppb, info dict (Q, noise ppb, floor, ...).
    The plume source is placed uniformly within +-centre_jitter_px (10 m px) of the chip centre with a sub-pixel offset."""
    B, _, H, W = x.shape; dev = x.device
    r = lambda *s: torch.rand(*s, generator=gen, device=dev)
    th = torch.where(torch.isfinite(theta), theta, r(B) * 2 * math.pi) + torch.deg2rad(torch.tensor(wind_jitter_deg, device=dev)) * (2 * r(B) - 1)
    si = (H - 1) / 2 + (2 * r(B) - 1) * centre_jitter_px; sj = (W - 1) / 2 + (2 * r(B) - 1) * centre_jitter_px
    ii = torch.arange(H, device=dev, dtype=torch.float32)[None, :, None]; jj = torch.arange(W, device=dev, dtype=torch.float32)[None, None, :]
    east = (jj - sj[:, None, None]) * 10.0; north = -(ii - si[:, None, None]) * 10.0
    c, s = torch.cos(th)[:, None, None], torch.sin(th)[:, None, None]
    xp = east * c + north * s; yp = -east * s + north * c                       # metres in the plume frame
    col = src[:, 1, None, None] + xp / 20.0; row = src[:, 0, None, None] - yp / 20.0
    Hc, Wc = canv.shape[-2:]
    grid = torch.stack([(2 * col + 1) / Wc - 1, (2 * row + 1) / Hc - 1], -1)
    base = F.grid_sample(canv.float(), grid, mode='bilinear', padding_mode='zeros', align_corners=False).clamp_min(0)  # (B,1,H,W)
    amf = 1 / torch.cos(torch.deg2rad(sza.clamp(0, 75))) + 1 / torch.cos(torch.deg2rad(vza.clamp(0, 30)))
    sig_r = robust_sigma(log_ratio(x)); sens = tm.sensitivity(sensor, amf).abs()
    noise_ppb = sig_r / sens.clamp_min(1e-9)
    if q_fixed is not None: Q = torch.full((B,), float(q_fixed), device=dev)
    else:
        cut = 3000.0 * noise_ppb / edge_ppb.clamp_min(1e-3)                     # keep the LES downwind cut below the noise
        hi = torch.minimum(torch.full_like(cut, q_max), cut).clamp_min(q_min)
        Q = torch.exp(math.log(q_min) + r(B) * (torch.log(hi) - math.log(q_min)))
    delta = base * (Q / 3000.0)[:, None, None, None]
    r11, r12 = tm.rel(sensor, amf, delta.flatten(1))
    xo = x.clone()
    xo[:, 4] = x[:, 4] * r11.view(B, H, W); xo[:, 5] = x[:, 5] * r12.view(B, H, W)
    floor = floor_sigma * noise_ppb
    label = (delta >= floor[:, None, None, None]).float()
    return xo, label, delta, dict(Q=Q, noise_ppb=noise_ppb, floor=floor, amf=amf, sens=sens, src_i=si, src_j=sj)

def enhancement_stats(x, y):
    """Per chip: mean of (log-ratio, MBMP) inside the mask minus the median outside. Same estimator for real and
    synthetic chips. x (B,8,H,W), y (B,1,H,W) -> dict of (B,) arrays (NaN where the mask is empty)."""
    out = {}
    for k, f in (('ratio', log_ratio), ('mbmp', mbmp)):
        z = f(x); vals = []
        for i in range(len(z)):
            m = y[i, 0] > 0.5
            vals.append(float(z[i, 0][m].mean() - z[i, 0][~m].median()) if m.sum() >= 4 and (~m).sum() > 100 else float('nan'))
        out[k] = np.array(vals)
    out['area_px'] = y.flatten(1).sum(1).cpu().numpy()
    return out

# --------------------------------------------------------------------------------------------- model
class MethaneNetMBMP(mf.MethaneNet):
    """MethaneNet with a 10th in-graph channel: MBMP (target minus reference), fp32, mean-removed, x20.
    Input is still (B,8,H,W). Adds 290 parameters (first conv 32x1x3x3 + 1 BN pair) to the 4.13 M."""
    def __init__(self, encoder='mobilenet_v2', weights=None, decoder=(128, 64, 32, 16, 8)):
        nn.Module.__init__(self)
        import segmentation_models_pytorch as smp
        self.norm = nn.BatchNorm2d(mf.IN_CH + 2)
        self.net = smp.Unet(encoder, encoder_weights=weights, in_channels=mf.IN_CH + 2, classes=1, decoder_channels=tuple(decoder))
    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            t11, t12, r11, r12 = x[:, 4:5], x[:, 5:6], x[:, 6:7], x[:, 7:8]
            eps = 1e-3
            ratio = torch.log((t12 + eps) / (t11 + eps)) - torch.log((r12 + eps) / (r11 + eps))
            ratio = (ratio - ratio.mean(dim=(2, 3), keepdim=True)) * 20.0
            def mb(b11, b12):
                c = (b11 * b12).sum((2, 3), keepdim=True) / ((b12 * b12).sum((2, 3), keepdim=True) + 1e-6)
                return (c * b12 - b11) / (b11 + 1e-3)
            m = mb(t11, t12) - mb(r11, r12)
            m = (m - m.mean(dim=(2, 3), keepdim=True)) * 20.0
            z = self.norm(torch.cat([x, ratio, m], 1))
        H, W = int(x.shape[2]), int(x.shape[3])
        ph, pw = (-H) % 32, (-W) % 32; pt, pl = ph // 2, pw // 2
        if ph or pw: z = F.pad(z, (pl, pw - pl, pt, ph - pt), mode='reflect')
        return self.net(z)[:, :, pt:pt + H, pl:pl + W]

def make_model(kind, encoder='mobilenet_v2'):
    return MethaneNetMBMP(encoder) if kind == 'mbmp' else mf.MethaneNet(encoder, weights=None)

def load_expanded(model, sd):
    """Load a 9-input-channel state dict into a model that may have more input channels: the overlapping slice is
    copied; new conv input weights start at 0 (so the extra channel starts switched off), new BN entries at identity."""
    own = model.state_dict(); new = {}; grown = []
    for k, v in own.items():
        s = sd[k]
        if s.shape == v.shape: new[k] = s
        else:
            t = v.clone()
            if k.endswith('running_var') or (k.startswith('norm.') and k.endswith('weight')): t.fill_(1.0)
            else: t.zero_()
            sl = tuple(slice(0, n) for n in s.shape); t[sl] = s; new[k] = t; grown.append(k)
    model.load_state_dict(new); return grown

def n_params(m): return sum(p.numel() for p in m.parameters())

# --------------------------------------------------------------------------------------------- loss
def enh_weight(E, y, lo=0.5, hi=2.0):
    """Per-chip enhancement weight in [lo, hi] on mask pixels: clip((E - q05)/(q95 - q05), 0, 1) over that chip's mask
    pixels. Units cancel (MBMP leaf or ppb). Chips with < 4 mask pixels or no E get weight 1."""
    w = torch.ones_like(y)
    for i in range(len(y)):
        m = y[i, 0] > 0.5
        if E is None or m.sum() < 4: continue
        v = E[i, 0][m].float()
        if not torch.isfinite(v).all(): v = torch.nan_to_num(v)
        q = torch.quantile(v, torch.tensor([0.05, 0.95], device=v.device))
        z = ((E[i, 0].float() - q[0]) / (q[1] - q[0]).clamp_min(1e-6)).clamp(0, 1)
        w[i, 0] = torch.where(m, lo + (hi - lo) * z, torch.ones_like(z))
    return w

def weighted_dice_bce(logits, y, w, pos_weight=3.0):
    logits = logits.float()
    pw = torch.where(y > 0.5, pos_weight * w, torch.ones_like(w))
    bce = F.binary_cross_entropy_with_logits(logits, y, weight=pw)
    p = torch.sigmoid(logits); inter = (p * y).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + y.sum((1, 2, 3))
    return bce + (1 - (2 * inter + 1) / (den + 1)).mean()

# ------------------------------------------------------------------------------------------- data
class SynScenes(torch.utils.data.Dataset):
    """Plume-free pretraining pairs + one of the scene's k assigned bank canvases (random each time).
    hard: optional bool per pair (indexed like XP); hard pairs are returned with flag True (kept clean in training)."""
    def __init__(self, XP, idx, assign, canvas, meta_arr, seed=0, hard=None):
        self.XP, self.idx, self.assign, self.canvas, self.meta = XP, np.asarray(idx), assign, canvas, meta_arr
        self.hard = np.zeros(len(XP), bool) if hard is None else np.asarray(hard, bool)
        self.seed = seed; self.rng = None
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        if self.rng is None:
            wi = torch.utils.data.get_worker_info(); self.rng = np.random.default_rng(self.seed + (wi.id + 1 if wi else 0) * 7919 + int(time.time()))
        j = int(self.idx[i]); b = int(self.assign[j, self.rng.integers(self.assign.shape[1])])
        return (torch.from_numpy(np.array(self.XP[j])), torch.from_numpy(np.array(self.canvas[b]))[None],
                torch.from_numpy(self.meta[j]), b, bool(self.hard[j]))

class WeightedStream(torch.utils.data.Sampler):
    """Endless weighted sampling with replacement, drawn in chunks (WeightedRandomSampler would draw all
    num_samples up front). weights: one per dataset position."""
    def __init__(self, weights, generator, chunk=4096):
        self.w = torch.as_tensor(np.asarray(weights, np.float64)); self.g = generator; self.chunk = chunk
        assert len(self.w) and bool((self.w >= 0).all()) and float(self.w.sum()) > 0
    def __len__(self): return 10 ** 8
    def __iter__(self):
        while True: yield from torch.multinomial(self.w, self.chunk, replacement=True, generator=self.g).tolist()

def scene_meta(rows):
    """Per pretraining pair: sza, vza, theta (wind direction, rad ccw from east; NaN if calm), sensor index."""
    u, v = rows['meteo:wind_u'].values, rows['meteo:wind_v'].values
    th = np.arctan2(v, u); th[np.hypot(u, v) < 0.1] = np.nan
    sen = rows['target:sensor'].map({s: i for i, s in enumerate(SENSORS)}).fillna(0).values
    return np.stack([rows['target:sza'].values, rows['target:vza'].values, th, sen], 1).astype(np.float32)

def joint_aug(x, y, E, gen):
    """Per-sample rot90 + horizontal flip applied identically to input, label and enhancement map."""
    B = x.shape[0]; dev = x.device
    k = torch.randint(0, 4, (B,), generator=gen, device=dev).tolist(); f = (torch.rand(B, generator=gen, device=dev) < 0.5).tolist()
    xs, ys, es = [], [], []
    for i in range(B):
        a, b, e = torch.rot90(x[i], k[i], (1, 2)), torch.rot90(y[i], k[i], (1, 2)), torch.rot90(E[i], k[i], (1, 2))
        if f[i]: a, b, e = a.flip(-1), b.flip(-1), e.flip(-1)
        xs.append(a); ys.append(b); es.append(e)
    return torch.stack(xs), torch.stack(ys), torch.stack(es)

# ------------------------------------------------------------------------------------- scene scores
@torch.no_grad()
def scene_scores(model, X, idx, dev, amp_dtype=None, bs=32, kind='pairs', workers=2):
    """Scene-level scores at the 20 m grid (2x2 mean-pooled probabilities).
    kind='pairs': X is an (N,8,200,200) pair cache (real plume-free pairs) -> score (max), area (px >= 0.5).
    kind='plumes': X is the 31-channel fine-tune cache; target vs bg0 plus the plume mask ->
                   score, area, score_in (max inside the 20 m mask), mask_px."""
    model.eval(); out = {k: [] for k in ('score', 'area', 'score_in', 'mask_px')}
    idx = np.asarray(idx)
    for s in range(0, len(idx), bs):
        j = np.sort(idx[s:s + bs])
        if kind == 'pairs': x = torch.from_numpy(np.array(X[j])).to(dev).float(); y = None
        else:
            a = torch.from_numpy(np.array(X[j, :mf.NCH_BASIC])).to(dev); x, y, _ = mf.eval_batch(a)
        with torch.autocast(dev.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            p = torch.sigmoid(model(x).float())
        pp = F.avg_pool2d(p, 2); out['score'].append(pp.flatten(1).amax(1).cpu()); out['area'].append((pp >= 0.5).flatten(1).sum(1).cpu())
        if y is not None:
            yy = F.avg_pool2d(y, 2) >= 0.5
            out['score_in'].append(torch.where(yy, pp, torch.zeros_like(pp)).flatten(1).amax(1).cpu()); out['mask_px'].append(yy.flatten(1).sum(1).cpu())
    return {k: torch.cat(v).numpy() for k, v in out.items() if v}

def average_precision(pos, neg):
    """AP for 'plume chip' vs 'plume-free pair' from scene scores (step-wise, like sklearn)."""
    s = np.concatenate([pos, neg]); t = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    o = np.argsort(-s, kind='mergesort'); s, t = s[o], t[o]
    tp = np.cumsum(t); fp = np.cumsum(1 - t); last = np.r_[np.where(np.diff(s))[0], len(s) - 1]
    P = tp[last] / (tp[last] + fp[last]); R = tp[last] / max(len(pos), 1)
    return float(np.sum(np.diff(np.r_[0, R]) * P)), (P, R)

def thr_at_fa(neg, fa):
    """Smallest threshold with at most floor(fa * n) negatives scoring >= it."""
    n = len(neg); allowed = int(np.floor(fa * n)); s = np.sort(neg)[::-1]
    return float(np.nextafter(s[allowed], np.inf)) if allowed < n else 0.0

def operating_points(pos, neg, fas=(0.10, 0.05, 0.01)):
    """pos: dict from scene_scores(kind='plumes'); neg: score array. Detection = score >= thr (any flag) and
    hit = score_in >= thr (a flagged 20 m pixel inside the plume mask)."""
    rows = {}
    for fa in fas:
        t = thr_at_fa(neg, fa)
        rows[f'{int(round(fa * 100))}%'] = dict(threshold=t, real_fa=float((neg >= t).mean()) if len(neg) else float('nan'),
                                              detect=float((pos['score'] >= t).mean()) if len(pos['score']) else float('nan'),
                                              hit=float((pos['score_in'] >= t).mean()) if len(pos['score_in']) else float('nan'))
    return rows

# ------------------------------------------------------------------------------------------ training
def real_batch(a, rows, gen, cfg, meth, ch4, dev):
    """Real fine-tune chips: fr_neg's recipe (target vs bg0, a fraction neg_p swapped for bg1-vs-bg0 plume-free pairs),
    no augmentation inside mf.train_batch so that the label, input and ch4 map are augmented together afterwards."""
    x, y = mf.train_batch(a, rows, gen, dict(cfg, aug=0), meth)
    if ch4 is not None:
        E = torch.from_numpy(np.nan_to_num(np.array(ch4[np.sort(rows.cpu().numpy())], dtype=np.float32)))
        order = np.argsort(np.argsort(rows.cpu().numpy()))         # back to batch order
        E = E[torch.as_tensor(order)][:, None].to(dev)
    else: E = torch.zeros_like(y)
    return x, y, E

def syn_batch(xp, canv, pm, bidx, can_src, edge, tm, gen, inj_p, keep_clean=None, **kw):
    """Plume-free pairs: a fraction inj_p get an injected plume (label from the detectability floor), the rest stay
    clean (label 0). keep_clean: optional bool (B,) on the same device; those pairs are never injected (hard negatives)."""
    x = xp.float(); B = x.shape[0]; dev = x.device
    inj = torch.rand(B, generator=gen, device=dev) < inj_p
    if keep_clean is not None: inj = inj & ~keep_clean.to(dev).bool()
    y = torch.zeros(B, 1, *x.shape[2:], device=dev); E = torch.zeros_like(y); info = None
    if inj.any():
        k = inj.nonzero()[:, 0]
        xi, yi, di, info = inject(x[k], canv[k], can_src[bidx[k]], pm[k, 2], pm[k, 0], pm[k, 1], pm[k, 3].long(),
                                  edge[bidx[k]], tm, gen, **kw)
        x = x.clone(); x[k] = xi; y[k] = yi; E[k] = di
    return x, y, E, inj, info

def train_syn(name, kind, Xft, meta_ft, ch4, XP, tr_pairs, assign, canvas, can_src, edge, pmeta, tm, dev, out_root, *,
              init_sd, XV=None, val_neg=None, epochs=12, lr=3e-4, bs_real=8, bs_syn=8, inj_p=0.7, neg_p=0.3,
              q_min=500., q_max=20000., floor_sigma=0.5, enh_weighting=True, amp=True, workers=2, seed=0,
              project='methane-se2-syn', skip_done=True, log=print, channels_last=True, max_steps=None,
              preview_every=3, pair_weights=None, hard=None, extra_cfg=None):
    """Mixed batches: bs_real real fine-tune chips + bs_syn plume-free pairs (inj_p of them with an injected plume).
    With the defaults (8 + 8, inj_p 0.7) that is 50 % real, 35 % injected, 15 % untouched, as in the plan.
    One epoch = one pass over the real train chips. Keeps final.pt and best.pt (best val hit rate at 5 % val FA).
    pair_weights: optional sampling weight per entry of tr_pairs (default uniform). hard: optional bool per XP row;
    hard pairs are never injected (hard negatives)."""
    import trackio
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    out = f'{out_root}/{name}'; os.makedirs(out, exist_ok=True)
    if skip_done and os.path.exists(f'{out}/final.pt') and os.path.exists(f'{out}/best.pt'):
        log(f'[{name}] final.pt exists, skipping (set SKIP_DONE=False to retrain)'); return out
    cfg = dict(name=name, kind=kind, epochs=epochs, lr=lr, bs_real=bs_real, bs_syn=bs_syn, inj_p=inj_p, neg_p=neg_p,
               q_min=q_min, q_max=q_max, floor_sigma=floor_sigma, enh_weighting=enh_weighting, seed=seed, amp=amp,
               n_pretrain_pairs=len(tr_pairs), resolution='10m', chip=200, weighted_sampling=pair_weights is not None,
               n_hard=int(np.asarray(hard, bool)[np.asarray(tr_pairs)].sum()) if hard is not None else 0, **(extra_cfg or {}))
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    tr = np.where(meta_ft.split == 'train')[0]; va = np.where(meta_ft.split == 'validation')[0]
    meth = torch.tensor(meta_ft[['bg:methane1', 'bg:methane2', 'bg:methane3']].fillna(1.0).values, device=dev, dtype=torch.float32)
    model = make_model(kind); grown = load_expanded(model, init_sd); model.to(dev)
    if grown: log(f'[{name}] expanded from the 9-channel init: {grown}')
    if channels_last and dev.type == 'cuda': model = model.to(memory_format=torch.channels_last)
    amp_dtype = mf.amp_dtype_for(dev) if amp else None
    scaler = torch.amp.GradScaler('cuda', enabled=amp_dtype == torch.float16)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    dl_r = mf.loader(Xft, tr, bs_real, True, workers, seed, drop_last=True)
    g = torch.Generator(); g.manual_seed(seed)
    ds = SynScenes(XP, tr_pairs, assign, canvas, pmeta, seed, hard=hard)
    sampler = (torch.utils.data.RandomSampler(ds, replacement=True, num_samples=10 ** 8, generator=g) if pair_weights is None
               else WeightedStream(pair_weights, g))
    dl_s = torch.utils.data.DataLoader(ds, batch_size=bs_syn, sampler=sampler, num_workers=workers,
                                       pin_memory=dev.type == 'cuda', persistent_workers=workers > 0, drop_last=True)
    it_s = iter(dl_s)
    can_src_t = torch.tensor(np.asarray(can_src), device=dev, dtype=torch.float32); edge_t = torch.tensor(np.asarray(edge), device=dev, dtype=torch.float32)
    spe = max(len(dl_r), 1) if not max_steps else min(len(dl_r), max_steps); steps = epochs * spe
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(1, s / max(1, steps)))))
    dv = mf.loader(Xft, va, 32, False, workers)
    trackio.init(project=project, name=name, config=cfg)
    kw = dict(q_min=q_min, q_max=q_max, floor_sigma=floor_sigma)
    hist = []; best = -1.0
    def preview(xb, yb, ep):
        n = min(6, len(xb)); f, ax = plt.subplots(3, n, figsize=(2.2 * n, 6.6), squeeze=False)
        with torch.no_grad(): p = torch.sigmoid(model(xb[:n]).float()).cpu()
        lr_ = log_ratio(xb[:n]).cpu()
        for i in range(n):
            ax[0, i].imshow(lr_[i, 0], cmap='RdBu', vmin=-0.05, vmax=0.05); ax[1, i].imshow(yb[i, 0].cpu(), cmap='gray')
            ax[2, i].imshow(p[i, 0], cmap='magma', vmin=0, vmax=1)
        for r_ in ax.ravel(): r_.axis('off')
        f.suptitle(f'{name} ep {ep}: synthetic chips. ln(B12/B11) target-ref / label / prediction', fontsize=9); f.tight_layout()
        fn = f'{out}/syn_preview_ep{ep:03d}.png'; f.savefig(fn, dpi=70); plt.close(f); return fn
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); ls = []; qs = []; noise = []; lab_px = []; xb_last = yb_last = None; n_hard = n_syn = 0
        for step, (a, rows) in enumerate(dl_r):
            if step >= spe: break
            a = a.to(dev, non_blocking=True); rows = rows.to(dev)
            xr, yr, Er = real_batch(a, rows, gen, dict(neg_p=neg_p), meth, ch4 if enh_weighting else None, dev)
            xp_, cv, pm, bidx, hd = next(it_s)
            xs, ys, Es, inj, info = syn_batch(xp_.to(dev, non_blocking=True), cv.to(dev, non_blocking=True), pm.to(dev),
                                              bidx.to(dev), can_src_t, edge_t, tm, gen, inj_p,
                                              keep_clean=hd.to(dev) if hard is not None else None, **kw)
            n_hard += int(hd.sum()); n_syn += len(hd)
            if info is not None:
                qs += info['Q'].tolist(); noise += info['noise_ppb'].tolist(); lab_px += ys[inj].flatten(1).sum(1).tolist()
            x = torch.cat([xr, xs]); y = torch.cat([yr, ys]); E = torch.cat([Er, Es])
            x, y, E = joint_aug(x, y, E, gen)
            w = enh_weight(E, y) if enh_weighting else torch.ones_like(y)
            if channels_last and dev.type == 'cuda': x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(dev.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                logits = model(x)
            loss = weighted_dice_bce(logits, y, w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            ls.append(loss.item()); xb_last, yb_last = xs.detach(), ys.detach()
        rec = dict(epoch=ep, train_loss=float(np.mean(ls)) if ls else float('nan'), lr=opt.param_groups[0]['lr'], epoch_s=time.time() - t0,
                   syn_Q_median_kgh=float(np.median(qs)) if qs else float('nan'),
                   syn_noise_ppb_median=float(np.median(noise)) if noise else float('nan'),
                   syn_label_px_median=float(np.median(lab_px)) if lab_px else float('nan'),
                   syn_label_empty_frac=float(np.mean(np.array(lab_px) == 0)) if lab_px else float('nan'),
                   syn_hard_frac=n_hard / max(n_syn, 1))
        c = mf.predict_counts(model, dv, dev, amp_dtype)
        for gname in ('10m', '20m'):
            rec.update({f'val{"" if gname == "10m" else "20"}/{k}': v for k, v in mf.at(c, 0.5, gname).items()})
        if XV is not None and val_neg is not None and len(val_neg):
            pos = scene_scores(model, Xft, va, dev, amp_dtype, kind='plumes'); neg = scene_scores(model, XV, val_neg, dev, amp_dtype)['score']
            op = operating_points(pos, neg, (0.05,))['5%']
            rec.update({'val/hit@fa5': op['hit'], 'val/detect@fa5': op['detect'], 'val/thr@fa5': op['threshold'],
                        'val/AP': average_precision(pos['score'], neg)[0], 'val/real_fa@0.5': float((neg >= 0.5).mean())})
        sel = rec.get('val/hit@fa5', rec['val/iou'])
        lg = dict(rec)
        if xb_last is not None and (ep % preview_every == 0 or ep in (1, epochs)): lg['syn/preview'] = trackio.Image(preview(xb_last, yb_last, ep))
        trackio.log(lg); hist.append(rec)
        sd = {k: v.contiguous() for k, v in model.state_dict().items()}
        if sel > best:
            best = sel; torch.save(sd, f'{out}/best.pt.tmp'); os.replace(f'{out}/best.pt.tmp', f'{out}/best.pt'); rec['best'] = True
        log(f'[{name}] ep {ep}/{epochs} loss {rec["train_loss"]:.4f} val IoU {rec["val/iou"]:.3f} '
            f'hit@5%FA {rec.get("val/hit@fa5", float("nan")):.3f} AP {rec.get("val/AP", float("nan")):.3f} '
            f'realFA@0.5 {rec.get("val/real_fa@0.5", float("nan")):.3f} | Q~{rec["syn_Q_median_kgh"]:.0f} kg/h noise~{rec["syn_noise_ppb_median"]:.0f} ppb '
            f'label~{rec["syn_label_px_median"]:.0f} px ({rec["epoch_s"]:.0f}s)')
    torch.save(sd, f'{out}/final.pt.tmp'); os.replace(f'{out}/final.pt.tmp', f'{out}/final.pt')
    json.dump(dict(config=cfg, history=hist, best_val_selection=best), open(f'{out}/history.json', 'w'), indent=1)
    trackio.finish()
    return out

# ------------------------------------------------------------------------------------------ evaluation
def load_model(kind, sd, dev):
    m = make_model(kind); m.load_state_dict(sd); return m.to(dev).eval()

def evaluate(name, model, Xft, subsets, XF, fa_sets, XV, val_neg, va_idx, dev, amp_dtype=None, workers=2, cal_fa=None):
    """subsets: {'all'|'seen'|'unseen': fine-tune test indices}; fa_sets: {name: indices into XF}.
    cal_fa (e.g. 0.05): also calibrate an operating threshold on the 2021 val plume-free pairs (the threshold where the val
    false-alarm rate is cal_fa) and report every test metric at it: pixel IoU/F1 per subset, scene detect / hit rates and
    the real 2024 false-alarm rate it actually gives (res['calibrated']). Returns (summary dict, raw score arrays)."""
    res = dict(model=name, pixel={}, scene={}, val={}); raw = dict(pos={}, neg={})
    have_val = XV is not None and val_neg is not None and len(val_neg) and len(va_idx)
    t_cal = None
    if have_val:   # val first: the calibrated threshold must exist before the test passes
        pos = scene_scores(model, Xft, va_idx, dev, amp_dtype, kind='plumes'); neg = scene_scores(model, XV, val_neg, dev, amp_dtype)['score']
        if cal_fa: t_cal = thr_at_fa(neg, cal_fa)
        kw = dict(thrs=list(mf.THRS) + [t_cal]) if t_cal is not None else {}   # older methane_fullres.py has no thrs argument
        c = mf.predict_counts(model, mf.loader(Xft, va_idx, 32, False, workers), dev, amp_dtype, **kw)
        res['val'] = dict(AP=average_precision(pos['score'], neg)[0], ops=operating_points(pos, neg), n_pos=int(len(va_idx)),
                          n_neg=int(len(neg)), pixel={g: mf.at(c, 0.5, g) for g in ('10m', '20m')})
        raw['val'] = dict(pos=pos, neg=neg)
        if t_cal is not None:
            res['calibrated'] = dict(val_fa_target=float(cal_fa), threshold=float(t_cal), pixel={}, scene={},
                                     val=dict(real_fa=float((neg >= t_cal).mean()), detect=float((pos['score'] >= t_cal).mean()),
                                              hit=float((pos['score_in'] >= t_cal).mean()), pixel={g: mf.at(c, t_cal, g) for g in ('10m', '20m')}))
    kw = dict(thrs=list(mf.THRS) + [t_cal]) if t_cal is not None else {}
    for sub, idx in subsets.items():
        if len(idx) == 0: continue
        c = mf.predict_counts(model, mf.loader(Xft, idx, 32, False, workers), dev, amp_dtype, **kw)
        res['pixel'][sub] = {g: mf.at(c, 0.5, g) for g in ('10m', '20m')}; res['pixel'][sub]['n'] = int(len(idx))
        if t_cal is not None: res['calibrated']['pixel'][sub] = {g: mf.at(c, t_cal, g) for g in ('10m', '20m')}
        raw['pos'][sub] = scene_scores(model, Xft, idx, dev, amp_dtype, kind='plumes')
    for fs, idx in fa_sets.items():
        raw['neg'][fs] = scene_scores(model, XF, idx, dev, amp_dtype)['score'] if len(idx) else np.zeros(0)
    for sub, pos in raw['pos'].items():
        for fs, neg in raw['neg'].items():
            if not len(neg): continue
            ap, _ = average_precision(pos['score'], neg)
            res['scene'][f'{sub}|{fs}'] = dict(n_pos=int(len(pos['score'])), n_neg=int(len(neg)), AP=ap,
                                               real_fa_at_0p5=float((neg >= 0.5).mean()),
                                               detect_at_0p5=float((pos['score'] >= 0.5).mean()),
                                               hit_at_0p5=float((pos['score_in'] >= 0.5).mean()),
                                               ops=operating_points(pos, neg))
            if t_cal is not None:
                res['calibrated']['scene'][f'{sub}|{fs}'] = dict(real_fa=float((neg >= t_cal).mean()), detect=float((pos['score'] >= t_cal).mean()),
                                                                 hit=float((pos['score_in'] >= t_cal).mean()))
    return res, raw

def go_no_go(cand, base, lat_ratio, contract_ok, params_cand, params_base, fa_key='8740', sub_unseen='unseen', calibrated=False):
    """The plan's release rule, evaluated on the val-selected candidate only. calibrated=True adds criterion 2b: the same
    IoU rule with each model at its own val-calibrated threshold (evaluate(..., cal_fa=...)), so a candidate that would
    ship with a calibrated threshold is judged there too."""
    def g(r, sub, fs, op, k): return r['scene'].get(f'{sub}|{fs}', {}).get('ops', {}).get(op, {}).get(k, float('nan'))
    d_hit = g(cand, sub_unseen, fa_key, '5%', 'hit') - g(base, sub_unseen, fa_key, '5%', 'hit')
    d_iou = cand['pixel'].get(sub_unseen, {}).get('20m', {}).get('iou', float('nan')) - base['pixel'].get(sub_unseen, {}).get('20m', {}).get('iou', float('nan'))
    d_val = cand['val'].get('ops', {}).get('5%', {}).get('hit', float('nan')) - base['val'].get('ops', {}).get('5%', {}).get('hit', float('nan'))
    d_all = g(cand, 'all', fa_key, '5%', 'hit') - g(base, 'all', fa_key, '5%', 'hit')
    rel_p = abs(params_cand - params_base) / params_base
    crit = {
        '1 unseen-site hit rate at real FA 5% improves by >= 0.10': (d_hit, bool(d_hit >= 0.10)),
        '2 unseen-site pixel IoU at 0.5 (20 m grid) drops by <= 0.01': (d_iou, bool(d_iou >= -0.01)),
        '3 2021 val shows the same direction (val hit rate at 5% val FA)': (d_val, bool(d_val > 0)),
        '4a ONNX input contract 1x8x200x200 unchanged': (contract_ok, bool(contract_ok)),
        '4b parameter count unchanged (within 0.01 %)': (rel_p, bool(rel_p <= 1e-4)),
        '4c CPU latency within 10 %': (lat_ratio, bool(np.isfinite(lat_ratio) and lat_ratio <= 1.10)),
        '5 no worse at matched detection on all test plumes (hit rate at real FA 5% not lower)': (d_all, bool(d_all >= 0)),
    }
    if calibrated:
        ci = lambda r: r.get('calibrated', {}).get('pixel', {}).get(sub_unseen, {}).get('20m', {}).get('iou', float('nan'))
        d_cal = ci(cand) - ci(base)
        ks = list(crit); i = ks.index(next(k for k in ks if k.startswith('2 '))) + 1
        crit = dict(list(crit.items())[:i] + [('2b unseen-site pixel IoU at each model\'s val-calibrated threshold (20 m grid) drops by <= 0.01',
                                              (d_cal, bool(d_cal >= -0.01)))] + list(crit.items())[i:])
    return all(v[1] for v in crit.values()), {k: dict(value=(float(v[0]) if not isinstance(v[0], bool) else v[0]), ok=v[1]) for k, v in crit.items()}

# ------------------------------------------------------------------------------------------ sanity gate
def sanity_gate(XP, gate_idx, assign, canvas, can_src, edge, pmeta, tm, Xft, meta_ft, real_idx, dev, out_png, *,
                q=3000.0, flux_band=(2000.0, 4500.0), floor_sigma='auto', floor_grid=(0.25, 0.5, 0.75, 1.0, 1.5, 2.0),
                ratio_tol=3.0, area_tol=4.0, seed=0, real_max=160):
    """Stage 1. Inject Q = 3000 kg/h into the gate scenes and compare with real MethaneSET plumes whose MARS flux is
    in flux_band, using the same estimator (mean in mask minus median outside) for ln(B12/B11) and MBMP.
    Checks: table units, sign of the dip, enhancement magnitude within x ratio_tol, label area within x area_tol.
    floor_sigma='auto' picks the label floor (x scene noise) whose median label area is closest to the real one."""
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    rep = {}; ok = True; why = []
    worst, units = tm.check_units(); rep['units'] = dict(max_rel_err_T1800_vs_bg=worst, with_ltoa_correction=tm.with_ltoa)
    if worst > 0.005: ok = False; why.append(f'transmittance table: T(1800) differs from the tabulated background by {worst:.2%}')
    amf0 = torch.tensor([2.3]); s0 = torch.tensor([0])
    r11, r12 = tm.rel(s0, amf0, torch.tensor([[1000.0]]))
    rep['units'].update(b12_rel_change_per_1000ppb_amf2p3=float(r12 - 1), b11_rel_change_per_1000ppb_amf2p3=float(r11 - 1),
                        b12_table_scale=float(tm.Tbg['b12'][0, 0]), note='B12 table ~64 scale cancels in T(1800+d)/T(1800)')
    # real plumes
    fl = meta_ft['detection:ch4_fluxrate'].values[real_idx]
    sel = np.asarray(real_idx)[(fl >= flux_band[0]) & (fl <= flux_band[1])]
    if len(sel) < 8: sel = np.asarray(real_idx); rep['real_note'] = f'fewer than 8 plumes in {flux_band} kg/h: using all {len(sel)}'
    if len(sel) > real_max: sel = np.random.default_rng(seed).choice(sel, real_max, replace=False)
    a = torch.from_numpy(np.array(X_sorted(Xft, sel, mf.NCH_BASIC))).to(dev); xr, yr, _ = mf.eval_batch(a)
    real = enhancement_stats(xr, yr); rep['real'] = dict(n=int(len(sel)), flux_kgh_median=float(np.median(meta_ft['detection:ch4_fluxrate'].values[np.sort(sel)])))
    # synthetic
    gidx = np.sort(np.asarray(gate_idx)); gen = torch.Generator(device=dev); gen.manual_seed(seed)
    xp = torch.from_numpy(np.array(XP[gidx])).to(dev).float(); b = torch.as_tensor(assign[gidx, 0], device=dev)
    cv = torch.from_numpy(np.array(canvas[np.sort(np.unique(assign[gidx, 0]))])).to(dev)
    lut = {int(v): i for i, v in enumerate(np.sort(np.unique(assign[gidx, 0])))}
    cvb = cv[torch.as_tensor([lut[int(v)] for v in assign[gidx, 0]], device=dev)][:, None]
    pm = torch.from_numpy(pmeta[gidx]).to(dev)
    src_t = torch.tensor(np.asarray(can_src), device=dev, dtype=torch.float32)[b]; edge_t = torch.tensor(np.asarray(edge), device=dev, dtype=torch.float32)[b]
    gst = gen.get_state()
    areas = {}
    for fs in floor_grid:
        gen.set_state(gst)
        _, yl, _, _ = inject(xp, cvb, src_t, pm[:, 2], pm[:, 0], pm[:, 1], pm[:, 3].long(), edge_t, tm, gen, q_fixed=q, floor_sigma=fs)
        areas[fs] = float(np.median(yl.flatten(1).sum(1).cpu().numpy()))
    real_area = float(np.median(real['area_px']))
    fsig = min(floor_grid, key=lambda f: abs(math.log((areas[f] + 1) / (real_area + 1)))) if floor_sigma == 'auto' else float(floor_sigma)
    gen.set_state(gst)
    xi, yi, di, info = inject(xp, cvb, src_t, pm[:, 2], pm[:, 0], pm[:, 1], pm[:, 3].long(), edge_t, tm, gen, q_fixed=q, floor_sigma=fsig)
    syn = enhancement_stats(xi, yi)
    true_dip = []  # exact injected signal (injected minus clean) inside the label, no noise
    lr_i, lr_c = log_ratio(xi), log_ratio(xp)
    for i in range(len(xi)):
        m = yi[i, 0] > 0.5; true_dip.append(float((lr_i[i, 0] - lr_c[i, 0])[m].mean()) if m.sum() else float('nan'))
    med = lambda v: float(np.nanmedian(v)) if np.isfinite(v).any() else float('nan')
    rep['label_floor'] = dict(chosen_sigma=fsig, mode=floor_sigma, median_area_px_by_sigma=areas, real_median_area_px=real_area)
    rep['synthetic'] = dict(n=int(len(gidx)), Q_kgh=q, noise_ppb_median=float(info['noise_ppb'].median()),
                            noise_ppb_iqr=[float(info['noise_ppb'].quantile(0.25)), float(info['noise_ppb'].quantile(0.75))],
                            floor_ppb_median=float(info['floor'].median()), peak_delta_ppb_median=float(di.flatten(1).amax(1).median()),
                            empty_label_frac=float(np.mean(syn['area_px'] == 0)), true_dip_ratio_median=med(np.array(true_dip)))
    for k in ('ratio', 'mbmp'):
        rep[f'{k}_enhancement'] = dict(real_median=med(real[k]), real_iqr=[float(np.nanpercentile(real[k], 25)), float(np.nanpercentile(real[k], 75))] if np.isfinite(real[k]).any() else None,
                                       syn_median=med(syn[k]), syn_iqr=[float(np.nanpercentile(syn[k], 25)), float(np.nanpercentile(syn[k], 75))] if np.isfinite(syn[k]).any() else None,
                                       syn_over_real=med(syn[k]) / med(real[k]) if med(real[k]) else float('nan'),
                                       syn_negative_frac=float(np.nanmean(syn[k] < 0)) if np.isfinite(syn[k]).any() else float('nan'),
                                       real_negative_frac=float(np.nanmean(real[k] < 0)) if np.isfinite(real[k]).any() else float('nan'))
    rep['area'] = dict(real_median_px=real_area, syn_median_px=float(np.median(syn['area_px'])),
                       syn_over_real=(float(np.median(syn['area_px'])) + 1) / (real_area + 1))
    # decisions
    rr = rep['ratio_enhancement']
    if not (rr['syn_negative_frac'] >= 0.8): ok = False; why.append(f"only {rr['syn_negative_frac']:.0%} of injected chips show a B12/B11 dip")
    for k in ('ratio', 'mbmp'):
        x = rep[f'{k}_enhancement']['syn_over_real']
        if not (np.isfinite(x) and 1 / ratio_tol <= x <= ratio_tol): ok = False; why.append(f'{k} enhancement synthetic/real = {x:.2f}, outside [1/{ratio_tol:g}, {ratio_tol:g}]')
    x = rep['area']['syn_over_real']
    if not (1 / area_tol <= x <= area_tol): ok = False; why.append(f'label area synthetic/real = {x:.2f}, outside [1/{area_tol:g}, {area_tol:g}]')
    if rep['synthetic']['empty_label_frac'] > 0.5: ok = False; why.append(f"{rep['synthetic']['empty_label_frac']:.0%} of 3 t/h injections leave an empty label")
    rep['ok'] = ok; rep['why'] = why
    # figure
    n = min(5, len(sel), len(gidx)); pick_r = np.linspace(0, len(sel) - 1, n).astype(int); pick_s = np.linspace(0, len(gidx) - 1, n).astype(int)
    f, ax = plt.subplots(3, 2 * n, figsize=(2.1 * 2 * n, 7.2), squeeze=False)
    lr_r = log_ratio(xr).cpu(); lr_s = lr_i.cpu(); xr_c, xi_c = xr.cpu(), xi.cpu()
    fl_s = meta_ft['detection:ch4_fluxrate'].values[np.sort(sel)]
    for i in range(n):
        for j, (L, X_, Y, t, k_) in enumerate(((lr_r, xr_c, yr.cpu(), f'real {fl_s[pick_r[i]] / 1000:.1f} t/h', pick_r[i]),
                                               (lr_s, xi_c, yi.cpu(), f'synthetic {q / 1000:.0f} t/h', pick_s[i]))):
            c_ = 2 * i + j; z = L[k_, 0] - L[k_, 0].median()
            ax[0, c_].imshow(np.clip(X_[k_, [2, 1, 0]].permute(1, 2, 0).numpy() * 3.5, 0, 1)); ax[0, c_].set_title(t, fontsize=7)
            ax[1, c_].imshow(z, cmap='RdBu', vmin=-0.04, vmax=0.04); ax[2, c_].imshow(Y[k_, 0], cmap='gray')
    for r_ in ax.ravel(): r_.axis('off')
    f.suptitle(f'Injection sanity: RGB / ln(B12/B11) target-ref (median removed, +-0.04) / label. Gate {"PASS" if ok else "FAIL"}', fontsize=9)
    f.tight_layout(); f.savefig(out_png, dpi=80); plt.close(f)
    f, ax = plt.subplots(1, 3, figsize=(12, 3.2))
    for a_, k, t in ((ax[0], 'ratio', 'ln(B12/B11) enhancement'), (ax[1], 'mbmp', 'MBMP enhancement')):
        rv, sv = real[k][np.isfinite(real[k])], syn[k][np.isfinite(syn[k])]; both = np.r_[rv, sv]
        bins = np.linspace(*np.percentile(both, [1, 99]), 40) if len(both) > 1 else 10
        a_.hist(rv, bins, alpha=.6, density=True, label=f'real {flux_band[0] / 1000:g}-{flux_band[1] / 1000:g} t/h (n={len(rv)})')
        a_.hist(sv, bins, alpha=.6, density=True, label=f'synthetic {q / 1000:g} t/h (n={len(sv)})'); a_.set_title(t, fontsize=9); a_.legend(fontsize=7)
    bins = np.linspace(0, 4.2, 40)
    ax[2].hist(np.log10(real['area_px'] + 1), bins, alpha=.6, density=True, label='real'); ax[2].hist(np.log10(syn['area_px'] + 1), bins, alpha=.6, density=True, label='synthetic')
    ax[2].set_title('log10(label area px + 1), 10 m', fontsize=9); ax[2].legend(fontsize=7)
    f.tight_layout(); f.savefig(out_png.replace('.png', '_hist.png'), dpi=80); plt.close(f)
    return ok, rep

def X_sorted(X, idx, nch=None):
    j = np.sort(np.asarray(idx)); return X[j, :nch] if nch else X[j]


# ------------------------------------------------------------------------------- round 2: sampling weights
def swir_albedo(X, idx, c0=4, step=8):
    """Mean target B11/B12 reflectance per chip on a coarse grid (every step-th pixel): a cheap surface-brightness
    proxy. Pair caches: c0 = 4 (channels 4, 5); the 31-channel fine-tune cache: c0 = 5. Reads only those two channels."""
    idx = np.asarray(idx); out = np.empty(len(idx), np.float32)
    for n, k in enumerate(idx):
        out[n] = np.nanmean(np.asarray(X[int(k), c0:c0 + 2, ::step, ::step], dtype=np.float32))
    return out

def unseen_like_weights(sites, countries, albedo, train_sites, *, novel_boost=2.0, a_site=0.5, a_country=0.5, a_albedo=0.5,
                        n_bins=10, w_cap=20.0, mix=0.5):
    """Sampling weights for plume-free training pairs that push the synthetic half toward conditions the fine-tune train
    split covers least. Uses only training-split information: no test or val statistics.
      * novel site (no fine-tune train chip at that site name) x novel_boost;
      * site balance  count(site)^-a_site      (big, often revisited sites stop dominating);
      * region balance count(country)^-a_country;
      * surface balance count(albedo bin)^-a_albedo (n_bins equal-width SWIR-brightness bins between the pool's p1 and p99).
    Weights are normalised to mean 1, capped at w_cap, then mixed with uniform: mix * w + (1 - mix), so at least
    (1 - mix) of the draws stay uniform over the pool. Returns (weights, report)."""
    sites = pd.Series(np.asarray(sites)); countries = pd.Series(np.asarray(countries)).fillna('unknown')
    albedo = np.asarray(albedo, np.float64); ok = np.isfinite(albedo)
    lo, hi = np.nanquantile(albedo[ok], [0.01, 0.99]) if ok.any() else (0.0, 1.0)
    edges = np.linspace(lo, hi, n_bins + 1)[1:-1]          # equal-width bins: rare (very dark / bright) surfaces get more weight
    abin = np.where(ok, np.searchsorted(edges, np.where(ok, albedo, 0)), n_bins)
    novel = ~sites.isin(set(train_sites)).values
    cnt = lambda s: s.map(s.value_counts()).values.astype(np.float64)
    w = (np.where(novel, novel_boost, 1.0) * cnt(sites) ** -a_site * cnt(countries) ** -a_country
         * cnt(pd.Series(abin)) ** -a_albedo)
    w = w / w.mean(); w = np.minimum(w, w_cap); w = w / w.mean(); w = mix * w + (1 - mix)
    rep = pool_report(sites, countries, albedo, novel, None) | {'weighted': pool_report(sites, countries, albedo, novel, w)}
    rep['effective_sample_size'] = float(w.sum() ** 2 / (w ** 2).sum()); rep['n'] = int(len(w))
    rep['settings'] = dict(novel_boost=novel_boost, a_site=a_site, a_country=a_country, a_albedo=a_albedo, n_bins=n_bins, w_cap=w_cap, mix=mix)
    return w, rep

def pool_report(sites, countries, albedo, novel, w=None, top=8):
    """Share of draws by country (top), novel-site share, distinct sites, albedo quartiles; weighted if w is given."""
    w = np.ones(len(sites)) if w is None else np.asarray(w, np.float64); w = w / w.sum()
    c = pd.Series(w).groupby(pd.Series(np.asarray(countries)).fillna('unknown').values).sum().sort_values(ascending=False)
    a = np.asarray(albedo, np.float64); ok = np.isfinite(a); o = np.argsort(a[ok]); cw = np.cumsum(w[ok][o]) / max(w[ok].sum(), 1e-12)
    aq = [float(a[ok][o][min(np.searchsorted(cw, p), ok.sum() - 1)]) for p in (0.25, 0.5, 0.75)] if ok.any() else [float('nan')] * 3
    return dict(country_share={k: round(float(v), 4) for k, v in c.head(top).items()}, novel_site_share=float(w[np.asarray(novel, bool)].sum()),
                n_sites=int(pd.Series(np.asarray(sites)).nunique()), albedo_q25_q50_q75=aq)

def hard_negatives(scores, top_frac=0.10):
    """Hard plume-free training pairs: the top_frac highest scene scores of the init model (fr_neg)."""
    scores = np.asarray(scores, np.float64); t = np.quantile(scores, 1 - top_frac)
    return scores >= t, float(t)

def boost_share(w, flag, share):
    """Scale the weights of flagged entries so they make up `share` of the weighted draws."""
    w = np.asarray(w, np.float64).copy(); f = np.asarray(flag, bool); Wf, Wn = w[f].sum(), w[~f].sum()
    if Wf > 0 and Wn > 0 and 0 < share < 1: w[f] *= share * Wn / ((1 - share) * Wf)
    return w / w.mean()

In [ ]:
%%writefile thresholds.py
"""Operating thresholds for the released full-res model (`fr_neg`): a two-stage read-out of the same network output.

Written by fr_neg_thresholds_colab.ipynb next to methane_fullres.py (`mf`) and synplume.py (`sp`). The onboard contract
is unchanged: `reflectance` float32 1x8x200x200 -> `plume_logits` 1x1x200x200. Only the read-out changes:

    p  = sigmoid(plume_logits)                 # 1x1x200x200, 10 m
    pp = avg_pool2d(p, 2)                      # 1x1x100x100, 20 m grid
    scene_score = max(pp)                      # same score as synplume.scene_scores
    plume_present = scene_score >= T_scene     # stage 1: T_scene set on 2021 val plume-free pairs (val FA target)
    mask = plume_present & (pp >= t_px)        # stage 2: t_px picked on 2021 val plume chips (pixel IoU)

The scene score is the MAXIMUM of ~10,000 pooled pixels, so its false-alarm-calibrated thresholds sit near 0.99. Used as a
pixel threshold they shrink every mask to its few most confident pixels (IoU ~0.03 in round 2). Hence two thresholds.

Mask metrics are micro-averaged over all chips of a split, like mf.predict_counts. "end-to-end": a chip that stage 1 does
not flag predicts an empty mask (all its plume pixels count as misses). "flagged only": the chips stage 1 flags.
"""
import numpy as np, torch, torch.nn.functional as F
import methane_fullres as mf

PX_GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)
GRIDS = {'20m': 10, '10m': 40}   # grid -> minimum flagged pixels for a chip (0.4 ha), as in mf.predict_counts


@torch.no_grad()
def chip_table(model, Xft, idx, dev, amp_dtype=None, bs=16, workers=2, px_grid=PX_GRID):
    """One pass over fine-tune chips `idx` (plume pair target vs bg0, and the chip's plume-free pair bg1 vs bg0).
    Returns numpy arrays, one row per chip in `idx` order:
      score, score_in, mask_px (20 m, as sp.scene_scores), n_score (scene score of the plume-free pair),
      tp_/fp_/fn_{grid}: (N, K) pixel counts at each px threshold, npx_{grid}: (N, K) flagged pixels on the plume-free pair."""
    model.eval(); thr = torch.tensor(np.asarray(px_grid, np.float32), device=dev)[None, :, None, None]
    out = {k: [] for k in ('idx', 'score', 'score_in', 'mask_px', 'n_score')}
    for g in GRIDS: out.update({f'tp_{g}': [], f'fp_{g}': [], f'fn_{g}': [], f'npx_{g}': []})
    for a, j in mf.loader(Xft, idx, bs, False, workers):
        a = a.to(dev, non_blocking=True); x, y, xn = mf.eval_batch(a)
        with torch.autocast(dev.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            p = torch.sigmoid(model(x).float()); pn = torch.sigmoid(model(xn).float())
        pp, ppn = F.avg_pool2d(p, 2), F.avg_pool2d(pn, 2); yy = F.avg_pool2d(y, 2) >= 0.5
        out['idx'].append(j); out['score'].append(pp.flatten(1).amax(1).cpu()); out['n_score'].append(ppn.flatten(1).amax(1).cpu())
        out['score_in'].append(torch.where(yy, pp, torch.zeros_like(pp)).flatten(1).amax(1).cpu()); out['mask_px'].append(yy.flatten(1).sum(1).cpu())
        for g, (q, t, qn) in {'20m': (pp, yy, ppn), '10m': (p, y >= 0.5, pn)}.items():
            b = q >= thr; bn = qn >= thr                       # (B, K, H, W)
            out[f'tp_{g}'].append((b & t).flatten(2).sum(2).cpu()); out[f'fp_{g}'].append((b & ~t).flatten(2).sum(2).cpu())
            out[f'fn_{g}'].append((~b & t).flatten(2).sum(2).cpu()); out[f'npx_{g}'].append(bn.flatten(2).sum(2).cpu())
    res = {k: torch.cat(v).numpy() for k, v in out.items() if v}
    if 'idx' in res: assert np.array_equal(res['idx'], np.asarray(idx)), 'loader order changed'
    res['px_grid'] = np.asarray(px_grid, np.float64)
    return res


def subset(tab, rows):
    """Rows of a chip table (rows: boolean mask or positions)."""
    return {k: (v if k == 'px_grid' else v[rows]) for k, v in tab.items()}


def _k(tab, t_px):
    k = int(np.argmin(np.abs(tab['px_grid'] - t_px)))
    assert abs(tab['px_grid'][k] - t_px) < 1e-6, f'pixel threshold {t_px} is not on the grid {tab["px_grid"]}'
    return k


def _prf(tp, fp, fn):
    if tp + fp + fn == 0: return dict(iou=np.nan, precision=np.nan, recall=np.nan, f1=np.nan)   # no chips / nothing to score
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(iou=tp / (tp + fp + fn), precision=P, recall=R, f1=2 * P * R / max(P + R, 1e-9))


def mask_metrics(tab, T_scene, t_px, grid='20m'):
    """Pixel metrics of the two-stage mask. end_to_end: unflagged chips predict nothing. flagged: only flagged chips.
    chip_fa: plume-free pairs (bg1 vs bg0) of the same chips that stage 1 flags AND that keep >= 0.4 ha of mask."""
    k = _k(tab, t_px); flag = tab['score'] >= T_scene
    tp, fp, fn = (tab[f'{c}_{grid}'][:, k].astype(np.float64) for c in ('tp', 'fp', 'fn'))
    e2e = _prf(tp[flag].sum(), fp[flag].sum(), fn[flag].sum() + (tp + fn)[~flag].sum())
    fl = _prf(tp[flag].sum(), fp[flag].sum(), fn[flag].sum()) if flag.any() else dict(iou=np.nan, precision=np.nan, recall=np.nan, f1=np.nan)
    nflag = tab['n_score'] >= T_scene
    return dict(end_to_end=e2e, flagged=fl, n_flagged=int(flag.sum()),
                chip_fa=float((nflag & (tab[f'npx_{grid}'][:, k] >= GRIDS[grid])).mean()) if len(flag) else np.nan,
                chip_fa_scene_only=float(nflag.mean()) if len(flag) else np.nan)


def scene_rates(tab, T_scene):
    if not len(tab['score']): return dict(detect=np.nan, hit=np.nan, n=0)
    return dict(detect=float((tab['score'] >= T_scene).mean()), hit=float((tab['score_in'] >= T_scene).mean()), n=int(len(tab['score'])))


def pick_px(val_tab, grid='20m', metric='iou'):
    """Val-optimal pixel threshold: best ungated val pixel IoU (stage 2 is judged on its own; 193 val chips, so noisy)."""
    vals = [_prf(*(val_tab[f'{c}_{grid}'][:, k].sum() for c in ('tp', 'fp', 'fn')))[metric] for k in range(len(val_tab['px_grid']))]
    k = int(np.nanargmax(vals)); return float(val_tab['px_grid'][k]), dict(zip(map(float, val_tab['px_grid']), map(float, vals)))


def cluster_boot(groups, stat, n=1000, seed=0, alpha=0.05):
    """Percentile CI of stat(rows) when whole clusters (sites) are resampled with replacement. NaN if < 2 clusters."""
    groups = np.asarray(groups); u, inv = np.unique(groups, return_inverse=True)
    if len(u) < 2 or n <= 0: return [float('nan'), float('nan')]
    members = [np.where(inv == c)[0] for c in range(len(u))]; rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        rows = np.concatenate([members[c] for c in rng.integers(0, len(u), len(u))]); vals.append(stat(rows))
    return [float(np.nanquantile(vals, alpha / 2)), float(np.nanquantile(vals, 1 - alpha / 2))]


def logit(p): p = float(p); return float(np.log(p / (1 - p))) if 0 < p < 1 else float('inf') if p >= 1 else float('-inf')

In [ ]:
sys.path.insert(0, os.getcwd())
import methane_fullres as mf; importlib.reload(mf)
import synplume as sp; importlib.reload(sp)
import thresholds as th; importlib.reload(th)
for f in ('methane_fullres.py', 'synplume.py', 'thresholds.py'): shutil.copy(f, f'{OUT}/{f}')

## 4. Fine-tune data (MethaneSET S2 finetune v1.4.0): 10 m cache, seen / unseen test sites
As round 2 section 5, without the `ch4` leaf. Reuses an existing `cache10m.npy` (`CACHE_PATH` or `FT_WORK/data`). Seen = a train chip has the same `site:location_name` and an emission point within 1 km.

In [ ]:
from huggingface_hub import hf_hub_download
t = time.time()
REPO, SUB = 'tacofoundation/methaneset', 'methaneset-s2-finetune'
FT_DATA = f'{FT_WORK}/data' if os.path.exists(f'{FT_WORK}/data') else DATA
CACHE = CACHE_PATH or (f'{FT_DATA}/cache10m.npy' if os.path.exists(f'{FT_DATA}/cache10m.npy') else f'{DATA}/cache10m.npy')
if not SMOKE_TEST: drive_get('cache10m.npy')
meta_files = {f: hf_hub_download(REPO, f'{SUB}/.tacocat/{f}', repo_type='dataset', local_dir=DATA) for f in ['level0.parquet', 'level1.parquet']}
l0_full = pd.read_parquet(meta_files['level0.parquet']).reset_index(drop=True)
l1 = pd.read_parquet(meta_files['level1.parquet'])
def hav(a, b, c, d):
    p = np.pi / 180; x = np.sin((c - a) * p / 2) ** 2 + np.cos(a * p) * np.cos(c * p) * np.sin((d - b) * p / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(x))
_tr = l0_full[l0_full.split == 'train']; _te = l0_full[l0_full.split == 'test']
SEEN = {r['id']: bool(((_tr['site:location_name'].values == r['site:location_name']) &
                       (hav(r['emission:lat'], r['emission:lon'], _tr['emission:lat'].values, _tr['emission:lon'].values) <= 1.0)).any())
        for _, r in _te.iterrows()}
UNSEEN_SITES = set(_te[~_te.id.map(SEEN)]['site:location_name'])
print(f'test chips {len(_te)}: unseen {sum(not v for v in SEEN.values())} at {len(UNSEEN_SITES)} sites')
l0 = l0_full
SMALL = ['Oman', 'Poland', 'Russian_Federation', 'Venezuela', 'Argentina', 'China', 'Jordan']
if SMOKE_TEST:
    l0 = l0[l0['internal:source_file'].isin([f'{SUB}_{c}.tacozip' for c in SMALL])].reset_index(drop=True)
    CACHE = f'{DATA}/cache10m_smoke.npy'
ZIPDIR = f'{FT_DATA}/{SUB}' if os.path.exists(f'{FT_DATA}/{SUB}') else f'{DATA}/{SUB}'
meta = l0[[c for c in mf.META_COLS if c in l0.columns]].copy()
if SMOKE_TEST: meta['split'] = [['train', 'train', 'validation', 'test'][i % 4] for i in range(len(meta))]
if not os.path.exists(CACHE):
    if not need_disk(1.0 if SMOKE_TEST else 15.1, 'fine-tune zips (6.2 GB) + 10 m cache (8.9 GB)'):
        raise SystemExit('Not enough disk for the fine-tune cache. Free space, or point CACHE_PATH / FT_WORK at an existing cache10m.npy.')
    for z in sorted(l0['internal:source_file'].unique()):
        if not os.path.exists(f'{ZIPDIR}/{z}'): hf_hub_download(REPO, f'{SUB}/{z}', repo_type='dataset', local_dir=os.path.dirname(ZIPDIR))
    mf.build_cache(l0, l1, ZIPDIR, CACHE, workers=os.cpu_count())
    if not SMOKE_TEST: drive_put('cache10m.npy')
    if DELETE_ZIPS_AFTER_USE and ZIPDIR.startswith(DATA) and os.path.isdir(ZIPDIR): shutil.rmtree(ZIPDIR); print('deleted the fine-tune zips: the cache is built')
XFT = mf.load_cache(CACHE, to_ram=False)
assert XFT.shape[1:] == (31, 200, 200) and len(XFT) == len(meta), (XFT.shape, len(meta))
va = np.where(meta.split == 'validation')[0]; te = np.where(meta.split == 'test')[0]
seen_flag = meta.id.map(SEEN)
TE = {'all': te, 'seen': te[seen_flag.values[te] == True], 'unseen': te[seen_flag.values[te] == False]}
if SMOKE_TEST and len(TE['unseen']) == 0: TE['unseen'] = te[::2]; TE['seen'] = te[1::2]; print('smoke: no real unseen chips in the slice, using every 2nd test chip as a stand-in')
SITE = meta['site:location_name'].astype(str).values
print('fine-tune val/test', len(va), len(te), '| test seen/unseen', len(TE['seen']), len(TE['unseen']), f'| cache {gb(CACHE):.1f} GB')
TIMES['finetune data'] = time.time() - t

## 5. Plume-free pairs (MethaneSET S2 pretraining): val and test false-alarm sets
Exactly the round-2 sets.
* **val FA** (chooses `T_scene`): 2021 pairs, not at unseen-test sites, a random `VAL_FA_N` (`random_state=0`).
* **test FA** (reporting only): 2024 pairs at sites in neither the fine-tune train split nor pretraining train (`real2024_strict`, 8,740; `real2024_strict_unseen` = those at unseen-test sites), plus every other 2024 pair at an unseen-test site (`real2024_unseen_all`, 3,066).

In [ ]:
t = time.time()
PSUB = 'methaneset-s2-pretraining'
pf = {f: hf_hub_download(REPO, f'{PSUB}/.tacocat/{f}', repo_type='dataset', local_dir=DATA) for f in ['level0.parquet', 'level1.parquet']}
p0 = pd.read_parquet(pf['level0.parquet']).reset_index(drop=True)
P1 = sp.leaf_index(pd.read_parquet(pf['level1.parquet'], columns=['id', 'internal:offset', 'internal:size', 'internal:source_file', 'internal:relative_path']))
atU = p0['site:location_name'].isin(UNSEEN_SITES)
ptr_all = p0[p0.split == 'train']
pva = p0[(p0.split == 'validation') & ~atU]
VAL_ROWS = pva.sample(min(VAL_FA_N, len(pva)), random_state=0).reset_index(drop=True)
pte = p0[p0.split == 'test']
TRAINED_SITES = set(l0_full[l0_full.split == 'train']['site:location_name']) | set(ptr_all['site:location_name'])
strict = pte[~pte['site:location_name'].isin(TRAINED_SITES)]
extraU = pte[pte['site:location_name'].isin(UNSEEN_SITES) & ~pte.index.isin(strict.index)]
print(f'2024 FA pairs: strict {len(strict)} (at unseen-test sites {strict["site:location_name"].isin(UNSEEN_SITES).sum()}, '
      f'{strict["site:location_name"].nunique()} sites) + {len(extraU)} more at unseen-test sites = {len(strict) + len(extraU)}')
if FA_N:
    k = min(FA_N, len(strict) + len(extraU)); ns = max(1, int(round(k * len(strict) / (len(strict) + len(extraU)))))
    strict = pd.concat([strict[strict['site:location_name'].isin(UNSEEN_SITES)].head(max(1, ns // 2)),
                        strict[~strict['site:location_name'].isin(UNSEEN_SITES)].head(ns - max(1, ns // 2))]); extraU = extraU.head(max(1, k - ns))
FA_ROWS = pd.concat([strict.assign(_strict=True), extraU.assign(_strict=False)]).reset_index(drop=True)
FA_ROWS['_unseen'] = FA_ROWS['site:location_name'].isin(UNSEEN_SITES)
assert set(FA_ROWS.split) == {'test'} and set(VAL_ROWS.split) == {'validation'}
PAIR_GB = 8 * 200 * 200 * 2 / 1e9; sfx = '_smoke' if SMOKE_TEST else ''
need = sum(0.0 if os.path.exists(f'{DATA}/{n}{sfx}_{len(r)}.npy') else len(r) * PAIR_GB for n, r in (('pairs_valfa', VAL_ROWS), ('pairs_testfa', FA_ROWS)))
if not need_disk(need, f'pair caches ({len(VAL_ROWS)} val FA + {len(FA_ROWS)} test FA)'): raise SystemExit('Not enough disk for the pair caches.')
RD = sp.LeafReader(local_dirs=[DATA]); RD.bytes = RD.remote = 0
CACHES = {}
for name, rows in (('pairs_valfa', VAL_ROWS), ('pairs_testfa', FA_ROWS)):
    p = f'{DATA}/{name}{sfx}_{len(rows)}.npy'
    if not SMOKE_TEST: drive_get(os.path.basename(p))
    CACHES[name] = sp.build_pair_cache(rows, P1, PSUB, p, RD, workers=READ_THREADS)
    if not SMOKE_TEST: drive_put(os.path.basename(p))
XV, bad_v = CACHES['pairs_valfa']; XF, bad_f = CACHES['pairs_testfa']
VAL_NEG = np.where(~bad_v)[0]
FA_SETS = {'real2024_strict': np.where(FA_ROWS._strict.values & ~bad_f)[0],
           'real2024_strict_unseen': np.where(FA_ROWS._strict.values & FA_ROWS._unseen.values & ~bad_f)[0],
           'real2024_unseen_all': np.where(FA_ROWS._unseen.values & ~bad_f)[0]}
TIMES['pair caches'] = time.time() - t; DL_GB = RD.bytes / 1e9
print({k: len(v) for k, v in FA_SETS.items()}, f'| val FA {len(VAL_NEG)} | downloaded {DL_GB:.2f} GB in {TIMES["pair caches"] / 60:.1f} min')

## 6. Weights
`WEIGHTS = 'released'`: the public `model.safetensors` of the released model, so the thresholds belong to exactly the file people download. Its SHA-256 is recorded.

In [ ]:
if WEIGHTS == 'released': WPATH = hf_hub_download(HF_MODEL, 'model.safetensors', local_dir=f'{DATA}/released')
else: WPATH = WEIGHTS
if WPATH.endswith('.safetensors'):
    from safetensors.torch import load_file; SD = load_file(WPATH)
else: SD = torch.load(WPATH, map_location='cpu')
W_SHA = hashlib.sha256(open(WPATH, 'rb').read()).hexdigest()
MODEL = sp.load_model('plain', SD, DEV)
AMP_DTYPE = mf.amp_dtype_for(DEV) if AMP else None
print('weights', WPATH, '| sha256', W_SHA[:16], '| params', sp.n_params(MODEL), '| precision', AMP_DTYPE or 'fp32')

## 7. Choose on val (2021) only
`T_scene` at each `VAL_FAS` target from the val plume-free pairs; `t_px` = best val pixel IoU. Written to `out/val_choice.json` **before** any test chip or 2024 pair is scored.

In [ ]:
t = time.time()
VAL_TAB = th.chip_table(MODEL, XFT, va, DEV, AMP_DTYPE, workers=WORKERS, px_grid=PX_GRID)
VAL_NEG_S = sp.scene_scores(MODEL, XV, VAL_NEG, DEV, AMP_DTYPE)['score']
T_SCENE = {f'{int(round(f * 100))}%': sp.thr_at_fa(VAL_NEG_S, f) for f in VAL_FAS}
T_PX, VAL_IOU_CURVE = th.pick_px(VAL_TAB, PX_PICK_GRID)
VAL_SUMMARY = {op: dict(T_scene=T, T_scene_logit_of_pooled_prob=th.logit(T), val_fa=float((VAL_NEG_S >= T).mean()), **th.scene_rates(VAL_TAB, T),
                        mask=th.mask_metrics(VAL_TAB, T, T_PX, PX_PICK_GRID)) for op, T in T_SCENE.items()}
VAL_SUMMARY['0.5'] = dict(T_scene=0.5, val_fa=float((VAL_NEG_S >= 0.5).mean()), **th.scene_rates(VAL_TAB, 0.5), mask=th.mask_metrics(VAL_TAB, 0.5, 0.5, PX_PICK_GRID))
VAL_CHOICE = dict(written=time.strftime('%Y-%m-%d %H:%M:%S %Z'), smoke_test=SMOKE_TEST, weights_sha256=W_SHA, n_val_plumes=int(len(va)), n_val_fa=int(len(VAL_NEG_S)),
                  T_scene=T_SCENE, t_px=T_PX, t_px_grid=PX_PICK_GRID, val_iou_by_t_px=VAL_IOU_CURVE, val=VAL_SUMMARY, precision=str(AMP_DTYPE or 'fp32'))
json.dump(VAL_CHOICE, open(f'{OUT}/val_choice.json', 'w'), indent=1, default=float)
print('T_scene (val FA target -> threshold):', {k: round(v, 5) for k, v in T_SCENE.items()}, f'| t_px = {T_PX} (val IoU {VAL_IOU_CURVE[T_PX]:.3f}; at 0.5: {VAL_IOU_CURVE.get(0.5, float("nan")):.3f})')
for op, v in VAL_SUMMARY.items():
    print(f'  val {op:>4s}: FA {v["val_fa"]:.3f} detect {v["detect"]:.3f} hit {v["hit"]:.3f} | mask IoU end-to-end {v["mask"]["end_to_end"]["iou"]:.3f}')
TIMES['val'] = time.time() - t

## 8. Test (2024): score once, then read every operating point
Rows: the current card (one threshold, 0.5), then each val-chosen `T_scene` with the val-chosen `t_px` (and with 0.5 for comparison). 95 % intervals resample **sites** (test plumes by `site:location_name`, plume-free pairs by their site).

In [ ]:
t = time.time()
TEST_TAB = th.chip_table(MODEL, XFT, te, DEV, AMP_DTYPE, workers=WORKERS, px_grid=PX_GRID)
pos_of = {i: k for k, i in enumerate(te)}
SUBTAB = {s: th.subset(TEST_TAB, np.array([pos_of[i] for i in idx], dtype=int)) for s, idx in TE.items()}
SUBSITE = {s: SITE[idx] for s, idx in TE.items()}
NEG = {fs: (sp.scene_scores(MODEL, XF, idx, DEV, AMP_DTYPE)['score'] if len(idx) else np.zeros(0)) for fs, idx in FA_SETS.items()}
NEG_SITE = {fs: FA_ROWS['site:location_name'].astype(str).values[idx] for fs, idx in FA_SETS.items()}
TIMES['test scoring'] = time.time() - t
OPS = [('0.5 (current card)', 0.5, 0.5)] + [(f'val {op} FA', T, T_PX) for op, T in T_SCENE.items()] + \
      ([(f'val {op} FA, mask 0.5', T, 0.5) for op, T in T_SCENE.items()] if T_PX != 0.5 else [])
def op_row(name, T, tp):
    r = dict(op=name, T_scene=float(T), t_px=float(tp), subsets={}, real_fa={})
    for s, tab in SUBTAB.items():
        sr = th.scene_rates(tab, T); m20 = th.mask_metrics(tab, T, tp, '20m'); m10 = th.mask_metrics(tab, T, tp, '10m')
        r['subsets'][s] = dict(**sr, iou20_end_to_end=m20['end_to_end']['iou'], iou20_flagged=m20['flagged']['iou'], f1_20_end_to_end=m20['end_to_end']['f1'],
                               precision20_end_to_end=m20['end_to_end']['precision'], recall20_end_to_end=m20['end_to_end']['recall'],
                               iou10_end_to_end=m10['end_to_end']['iou'], n_flagged=m20['n_flagged'], chip_fa_bg1=m20['chip_fa'], chip_fa_bg1_scene_only=m20['chip_fa_scene_only'],
                               hit_ci95=th.cluster_boot(SUBSITE[s], lambda rows, tab=tab: th.scene_rates(th.subset(tab, rows), T)['hit'], BOOT_N),
                               iou20_end_to_end_ci95=th.cluster_boot(SUBSITE[s], lambda rows, tab=tab: th.mask_metrics(th.subset(tab, rows), T, tp, '20m')['end_to_end']['iou'], BOOT_N))
    for fs, neg in NEG.items():
        if len(neg): r['real_fa'][fs] = dict(rate=float((neg >= T).mean()), n=int(len(neg)),
                                             ci95=th.cluster_boot(NEG_SITE[fs], lambda rows, neg=neg: float((neg[rows] >= T).mean()), BOOT_N))
    return r
RESULTS = [op_row(*o) for o in OPS]
TIMES['test metrics'] = time.time() - t - TIMES['test scoring']
rows = []
for r in RESULTS:
    for s, v in r['subsets'].items():
        rows.append(dict(op=r['op'], T_scene=r['T_scene'], t_px=r['t_px'], plumes=s, n=v['n'], detect=v['detect'], hit=v['hit'],
                         hit_lo=v['hit_ci95'][0], hit_hi=v['hit_ci95'][1], iou20_e2e=v['iou20_end_to_end'], iou_lo=v['iou20_end_to_end_ci95'][0],
                         iou_hi=v['iou20_end_to_end_ci95'][1], iou20_flagged=v['iou20_flagged'], iou10_e2e=v['iou10_end_to_end'], chip_fa_bg1=v['chip_fa_bg1'],
                         real_fa_strict=r['real_fa'].get('real2024_strict', {}).get('rate', np.nan), real_fa_unseen_all=r['real_fa'].get('real2024_unseen_all', {}).get('rate', np.nan)))
TABLE = pd.DataFrame(rows); TABLE.to_csv(f'{OUT}/thresholds_table.csv', index=False, float_format='%.4f')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print(TABLE.round(3).to_string(index=False))

## 9. Reproduction check against round 2
The round-2 run (`syn_results.json`, Colab T4, AMP fp16, `fr_neg/final.pt`) recorded these `fr_neg` numbers. Small differences are expected from fp32 here vs fp16 there; a large one means different weights or data.

In [ ]:
R2 = {'val T_scene 10%': 0.9943740963935852, 'val T_scene 5%': 0.9966223239898682, 'val T_scene 1%': 0.9983797669410706,
      'test IoU@0.5 20m all': 0.5206476378975718, 'test IoU@0.5 20m unseen': 0.4354487470015365,
      'real FA@0.5 real2024_strict': 0.3562929061784897, 'hit@0.5 unseen': 0.7370517928286853,
      'hit@val5% unseen': 0.3107569721115538, 'real FA@val5% real2024_strict': 0.028146453089244853}
r05 = RESULTS[0]; r5 = next(r for r in RESULTS if r['op'] == 'val 5% FA')
NOW = {'val T_scene 10%': T_SCENE.get('10%'), 'val T_scene 5%': T_SCENE.get('5%'), 'val T_scene 1%': T_SCENE.get('1%'),
       'test IoU@0.5 20m all': r05['subsets']['all']['iou20_end_to_end'], 'test IoU@0.5 20m unseen': r05['subsets']['unseen']['iou20_end_to_end'],
       'real FA@0.5 real2024_strict': r05['real_fa'].get('real2024_strict', {}).get('rate'), 'hit@0.5 unseen': r05['subsets']['unseen']['hit'],
       'hit@val5% unseen': r5['subsets']['unseen']['hit'], 'real FA@val5% real2024_strict': r5['real_fa'].get('real2024_strict', {}).get('rate')}
REPRO = {k: dict(round2=v, now=NOW[k], diff=(NOW[k] - v) if NOW[k] is not None else None) for k, v in R2.items()}
for k, v in REPRO.items(): print(f'{k:32s} round2 {v["round2"]:.4f} | now {v["now"] if v["now"] is None else round(v["now"], 4)} | diff {v["diff"] if v["diff"] is None else round(v["diff"], 4)}')
REPRO_OK = all(v['diff'] is not None and abs(v['diff']) <= (0.0005 if 'T_scene' in k else 0.01) for k, v in REPRO.items())
print('reproduces round 2:', REPRO_OK, '(smoke test: not expected)' if SMOKE_TEST else '')

## 10. Plots

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
neg = NEG['real2024_strict']
f, ax = plt.subplots(1, 3, figsize=(15, 4))
for s, c in zip(('all', 'seen', 'unseen'), ('k', 'tab:blue', 'tab:red')):
    if not len(neg) or not len(SUBTAB[s]['score']): continue
    ts = np.unique(np.r_[np.quantile(neg, np.linspace(0, 1, 300)), 0.5, list(T_SCENE.values())])
    ax[0].plot([(neg >= x).mean() for x in ts], [(SUBTAB[s]['score_in'] >= x).mean() for x in ts], color=c, label=f'{s} test plumes')
    for op, T in T_SCENE.items(): ax[0].plot([(neg >= T).mean()], [(SUBTAB[s]['score_in'] >= T).mean()], 'o', color=c, ms=5)
    ax[0].plot([(neg >= 0.5).mean()], [(SUBTAB[s]['score_in'] >= 0.5).mean()], 's', color=c, ms=5)
ax[0].set_xscale('log'); ax[0].set_xlabel('real 2024 false-alarm rate (real2024_strict)'); ax[0].set_ylabel('scene hit rate')
ax[0].set_title('dots: val-chosen T_scene (10/5/1 %), squares: 0.5', fontsize=9); ax[0].legend(fontsize=8)
for s, c in zip(('all', 'seen', 'unseen'), ('k', 'tab:blue', 'tab:red')):
    ax[1].plot(PX_GRID, [th.mask_metrics(SUBTAB[s], -1, x, '20m')['end_to_end']['iou'] for x in PX_GRID], color=c, label=f'test {s}')
ax[1].plot(list(VAL_IOU_CURVE), list(VAL_IOU_CURVE.values()), 'g--', label='val (chooses t_px)'); ax[1].axvline(T_PX, color='g', lw=.6)
ax[1].set_xlabel('pixel threshold t_px (20 m, no scene gate)'); ax[1].set_ylabel('pixel IoU'); ax[1].legend(fontsize=8)
Ts = np.r_[0.5, np.linspace(0.98, 0.9995, 40)]
for s, c in zip(('all', 'unseen'), ('k', 'tab:red')):
    ax[2].plot(Ts[1:], [th.mask_metrics(SUBTAB[s], T, T_PX, '20m')['end_to_end']['iou'] for T in Ts[1:]], color=c, label=f'{s}: end-to-end')
    ax[2].plot(Ts[1:], [th.mask_metrics(SUBTAB[s], T, T_PX, '20m')['flagged']['iou'] for T in Ts[1:]], color=c, ls=':', label=f'{s}: flagged chips only')
for T in T_SCENE.values(): ax[2].axvline(T, color='grey', lw=.5, ls='--')
ax[2].set_xlabel(f'T_scene (mask at t_px = {T_PX})'); ax[2].set_ylabel('pixel IoU (20 m)'); ax[2].legend(fontsize=7)
f.tight_layout(); f.savefig(f'{OUT}/thresholds.png', dpi=90); plt.close(f)
from IPython.display import Image, display; display(Image(f'{OUT}/thresholds.png'))

## 11. Write `out/thresholds.json` and `out/config_thresholds.json`
`config_thresholds.json` is a snippet for the model's `config.json`; nothing is uploaded anywhere.

In [ ]:
CONFIG = {'threshold': 0.5,
          'readout': 'p = sigmoid(plume_logits); pp = avg_pool2d(p, 2) (20 m); scene_score = max(pp); plume_present = scene_score >= scene_threshold; '
                     'mask = (pp >= mask_threshold) if plume_present else empty',
          'scene_threshold': T_SCENE.get(RECOMMENDED), 'mask_threshold': T_PX, 'mask_grid': '20m (2x2 mean-pooled probabilities)', 'min_mask_pixels_20m': 10,
          'operating_points': {op: {'scene_threshold': T, 'val_false_alarm_target': float(op.rstrip('%')) / 100,
                                    'test_real2024_strict_false_alarm': next(r for r in RESULTS if r['op'] == f'val {op} FA')['real_fa'].get('real2024_strict', {}).get('rate')}
                               for op, T in T_SCENE.items()},
          'recommended_operating_point': RECOMMENDED,
          'note': 'threshold (0.5) is the legacy single pixel threshold. Scene thresholds were chosen on 2021 val plume-free pairs only; '
                  'they are for scene_score (max of pooled probabilities), not for single pixels.'}
TIMES['total'] = time.time() - T0
SUMMARY = dict(notebook='fr_neg_thresholds_colab.ipynb', smoke_test=SMOKE_TEST, gpu=GPU, precision=str(AMP_DTYPE or 'fp32'), weights=WEIGHTS, weights_sha256=W_SHA,
               val_choice=VAL_CHOICE, test=RESULTS, reproduction_vs_round2=REPRO, reproduces_round2=REPRO_OK, config_snippet=CONFIG,
               n=dict(val_plumes=int(len(va)), val_fa=int(len(VAL_NEG_S)), test=dict((s, int(len(i))) for s, i in TE.items()), test_fa=dict((k, int(len(v))) for k, v in FA_SETS.items())),
               boot_n=BOOT_N, times_s=TIMES, downloaded_gb=DL_GB,
               caveats=['temporal split (val 2021, test 2024), not site-grouped: 799 of 1,050 test plumes are at sites seen in training',
                        'val FA pairs (2021, any non-unseen-test site) differ from the test FA pairs (2024, sites in no training split): the val FA rate need not transfer',
                        't_px picked on 193 val plume chips; T_scene at 1 % rests on ~20 val plume-free pairs above it',
                        'intervals are site-clustered percentile bootstraps over the test set only; they do not cover the val choice'])
json.dump(SUMMARY, open(f'{OUT}/thresholds.json', 'w'), indent=1, default=float)
json.dump(CONFIG, open(f'{OUT}/config_thresholds.json', 'w'), indent=1, default=float)
print(json.dumps(CONFIG, indent=1, default=float))
print({k: round(v / 60, 1) for k, v in TIMES.items()}, 'min')
print(f'wrote {OUT}/thresholds.json  <- send this file')